In [ ]:

!apt-get update -qq && apt-get install -y -qq libgomp1
!pip -q install --no-deps scikit-learn==1.5.2 lightgbm==4.5.0 xgboost==2.1.1 catboost==1.2.7 ruptures==1.1.9 hdbscan==0.8.33 plotly==5.24.1
import os, json, math, warnings, pathlib, numpy as np, pandas as pd, requests
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance

import lightgbm as lgb
try:
    import xgboost as xgb
except Exception:
    xgb = None  # e.g. OpenMP/libgomp missing on Colab; pipeline continues with RF+LGBM+CatBoost
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None  # e.g. numpy ABI mismatch on Colab; pipeline continues with RF+LGBM(+XGB)
import ruptures as rpt
try:
    import hdbscan
except Exception:
    hdbscan = None  # e.g. numpy ABI mismatch on Colab; anomaly ensemble uses IF+OCSVM+LOF only
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


BASE   = pathlib.Path("/content/live_project")
BRONZE = BASE/"data"/"bronze";   BRONZE.mkdir(parents=True, exist_ok=True)
SILVER = BASE/"data"/"silver";   SILVER.mkdir(parents=True, exist_ok=True)
GOLD   = BASE/"data"/"gold"  ;   GOLD.mkdir(parents=True, exist_ok=True)
ART    = BASE/"artifacts"    ;   ART.mkdir(parents=True, exist_ok=True)
INT    = BASE/"data"/"internal"; INT.mkdir(parents=True, exist_ok=True)
BUART  = ART/"bu";               BUART.mkdir(parents=True, exist_ok=True)
SCARDS = BUART/"scorecards";     SCARDS.mkdir(parents=True, exist_ok=True)
UPG    = ART/"upgrade";          UPG.mkdir(parents=True, exist_ok=True)
HISTORY = BASE/"history"  # optional; used by upgrade pack

# ----------------------- watchlist -------------------
ISO3 = ["DEU","NLD","POL","FRA","ITA","ESP","CZE","AUT","SWE","DNK","NOR","FIN",
        "GBR","IRL","USA","CAN","MEX","BRA","CHN","JPN","KOR","IND","THA","VNM","TUR"]
ISO2_TO_ISO3 = {
    "DE":"DEU","NL":"NLD","PL":"POL","FR":"FRA","IT":"ITA","ES":"ESP","CZ":"CZE","AT":"AUT","SE":"SWE",
    "DK":"DNK","NO":"NOR","FI":"FIN","GB":"GBR","UK":"GBR","IE":"IRL","US":"USA","CA":"CAN","MX":"MEX",
    "BR":"BRA","CN":"CHN","JP":"JPN","KR":"KOR","IN":"IND","TH":"THA","VN":"VNM","TR":"TUR"
}

print("UTC:", datetime.utcnow().strftime("%Y-%m-%d %H:%M"))

# ----------------------- World Bank (live, no key) -----------------------
WB_BASE = "https://api.worldbank.org/v2/country/{ISO}/indicator/{IND}?format=json"
def wb_fetch_indicator(iso_list, indicator, mrv=1):
    rows=[]
    for iso in iso_list:
        try:
            r = requests.get(WB_BASE.format(ISO=iso, IND=indicator),
                             params={"mrv":str(mrv)}, timeout=45)
            r.raise_for_status()
            js=r.json()
            if isinstance(js, list) and len(js)==2 and js[1]:
                for rec in js[1]:
                    if rec and rec.get("value") is not None:
                        rows.append({
                            "iso3": rec.get("countryiso3code"),
                            "date": int(rec.get("date")) if rec.get("date") else None,
                            "indicator": indicator,
                            "value": float(rec.get("value"))
                        })
        except Exception:
            pass
    return pd.DataFrame(rows)

LPI = "LP.LPI.OVRL.XQ"   # Logistics Performance Index
POP = "SP.POP.TOTL"      # Population
WGI_CODES = ["GE.EST","RQ.EST","RL.EST","CC.EST"]  # governance subindices (estimates)

print("Pulling World Bank…")
wb_lpi = wb_fetch_indicator(ISO3, LPI, mrv=1)
wb_pop = wb_fetch_indicator(ISO3, POP, mrv=1)
wb_wgi = pd.concat([wb_fetch_indicator(ISO3, c, mrv=1) for c in WGI_CODES], ignore_index=True)

for name, df in [("wb_lpi", wb_lpi), ("wb_pop", wb_pop), ("wb_wgi", wb_wgi)]:
    if not df.empty: df.to_parquet(BRONZE/f"{name}.parquet", index=False)

def latest_value(df, indicator, alias):
    if df.empty: return pd.DataFrame({"iso3": ISO3, alias: np.nan})
    d = df[df["indicator"]==indicator].copy()
    if d.empty: return pd.DataFrame({"iso3": ISO3, alias: np.nan})
    d = d.sort_values(["iso3","date"]).groupby("iso3").tail(1)
    out = d[["iso3","value"]].rename(columns={"value": alias})
    missing = [c for c in ISO3 if c not in set(out["iso3"])]
    if missing: out = pd.concat([out, pd.DataFrame({"iso3": missing, alias: np.nan})], ignore_index=True)
    return out

lpi_latest = latest_value(wb_lpi, LPI, "LPI")
pop_latest = latest_value(wb_pop, POP, "POP")

if not wb_wgi.empty:
    wgi = (wb_wgi.sort_values(["iso3","date"])
                 .groupby(["iso3","indicator"]).tail(1)
                 .pivot(index="iso3", columns="indicator", values="value")
                 .reset_index())
    for c in WGI_CODES:
        if c not in wgi.columns: wgi[c]=np.nan
    wgi["WGI_MEAN"] = wgi[WGI_CODES].mean(axis=1)
    wgi = wgi[["iso3","WGI_MEAN"]]
else:
    wgi = pd.DataFrame({"iso3": ISO3, "WGI_MEAN": np.nan})

# ----------------------- GDELT (live, no key) -----------------------
QUERY = ("(port OR strike OR flood OR storm OR customs OR wildfire OR landslide OR earthquake OR cyber) "
         "AND (delay OR disruption OR shutdown OR congestion OR outage OR blockade)")

def gdelt_timeline_matrix(query, timespan="30d"):
    try:
        r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc",
                         params={"query": query, "mode":"timelinesourcecountry",
                                 "timespan": timespan, "format":"JSON"},
                         timeout=45)
        r.raise_for_status()
        js = r.json()
        tl = js.get("timeline", [])
        return pd.DataFrame(tl) if tl else pd.DataFrame()
    except Exception:
        return pd.DataFrame()

def ensure_date_series(df: pd.DataFrame) -> pd.Series:
    for cand in ("date","datetime","time","timestamp"):
        if cand in df.columns:
            s = pd.to_datetime(df[cand].astype(str), errors="coerce")
            return s.dt.normalize()
    end = pd.Timestamp.utcnow().normalize()
    start = end - pd.Timedelta(days=max(len(df)-1, 0))
    return pd.Series(pd.date_range(start, end, freq="D"), index=df.index)

print("Pulling GDELT 30d timeline…")
raw = gdelt_timeline_matrix(QUERY, "30d")

# Build 30d matrix (raw + per-million)
if raw.empty:
    dates = pd.date_range(pd.Timestamp.utcnow().normalize()-pd.Timedelta(days=29),
                          pd.Timestamp.utcnow().normalize(), freq="D")
    mat = pd.DataFrame(0.0, index=dates, columns=ISO3)
else:
    raw["date"] = ensure_date_series(raw)
    iso2_cols = [c for c in raw.columns if c != "date"]
    long = raw.melt(id_vars=["date"], value_vars=iso2_cols, var_name="iso2", value_name="count").fillna(0)
    long["iso3"] = long["iso2"].str.upper().map(ISO2_TO_ISO3)
    long = long.dropna(subset=["iso3"])
    long = long[long["iso3"].isin(ISO3)]
    mat = long.pivot_table(index="date", columns="iso3", values="count", aggfunc="sum").sort_index()
    for c in ISO3:
        if c not in mat.columns: mat[c]=0.0
    end = mat.index.max() if len(mat.index) else pd.Timestamp.utcnow().normalize()
    start = end - pd.Timedelta(days=29)
    mat = mat.reindex(pd.date_range(start, end, freq="D"), fill_value=0.0)
    mat = mat[ISO3]

mat.astype(float).to_parquet(SILVER/"gdelt_counts_30d.parquet")

# per-million
POP_MAP = dict(zip(pop_latest["iso3"], pop_latest["POP"])) if not pop_latest.empty else {}
divisors = pd.Series({c: (POP_MAP.get(c, np.nan)/1_000_000.0) for c in mat.columns})
if divisors.notna().any():
    fill_val = float(divisors.dropna().median())
    divisors = divisors.fillna(fill_val if not np.isnan(fill_val) else 1.0)
else:
    divisors = pd.Series(1.0, index=mat.columns)
mat_pm = mat.divide(divisors, axis=1).astype(float)
mat_pm.to_parquet(SILVER/"gdelt_counts_pm_30d.parquet")

# 7d spike vs 30d baseline
pm_7d  = mat_pm.rolling(window=7,  min_periods=1).sum().iloc[-1]
pm_30m = mat_pm.rolling(window=30, min_periods=7).mean().iloc[-1]
pm_30s = mat_pm.rolling(window=30, min_periods=7).std(ddof=0).iloc[-1]
eps = 1e-6
news_7d_pm        = pm_7d
news_7d_z_pm      = (pm_7d - (pm_30m*7.0)) / (np.sqrt((pm_30s**2)*7.0) + eps)
news_7d_delta_pct = (pm_7d - (pm_30m*7.0)) / ((pm_30m*7.0) + eps)

news_feats = pd.DataFrame({
    "iso3": mat_pm.columns,
    "news_7d_pm": news_7d_pm.values.astype(float),
    "news_7d_z_pm": news_7d_z_pm.values.astype(float),
    "news_7d_delta_pct": news_7d_delta_pct.values.astype(float),
}).sort_values("iso3").reset_index(drop=True)
news_feats.to_csv(SILVER/"news_feats_gdelt_pm.csv", index=False)

# ----------------------- change-points -----------------------
def changepoint_score(series: pd.Series) -> float:
    y = series.values.astype(float)
    if np.allclose(y, y[0]) or len(y)<15:
        return 0.0
    try:
        algo = rpt.Pelt(model="rbf").fit(y)
        bkps = algo.predict(pen=1.0)
        if len(bkps) <= 1:
            return 0.0
        last_cp = sorted(bkps[:-1])[-1]
        pre = y[max(0,last_cp-7):last_cp]
        post = y[last_cp:min(len(y), last_cp+7)]
        if len(pre)==0 or len(post)==0:
            return 0.0
        diff = np.mean(post) - np.mean(pre)
        std  = np.std(y) if np.std(y)>0 else 1.0
        return float(diff / std)
    except Exception:
        return 0.0

cp_scores = {c: changepoint_score(mat_pm[c]) for c in mat_pm.columns}
cp_df = pd.DataFrame({"iso3": list(cp_scores.keys()), "cp_score": list(cp_scores.values())})
cp_df.to_csv(SILVER/"gdelt_changepoint_scores.csv", index=False)

# ----------------------- feature table -----------------------
def z(col):
    sd = col.std()
    return (col - col.mean()) / (sd if (sd and sd>0) else 1.0)

feat = (lpi_latest.merge(wgi, on="iso3", how="left")
                 .merge(pop_latest, on="iso3", how="left")
                 .merge(news_feats, on="iso3", how="left")
                 .merge(cp_df, on="iso3", how="left"))

feat["LPI_z"] = z(feat["LPI"])
feat["WGI_MEAN_z"] = z(feat["WGI_MEAN"])
feat["composite_risk_v1"] = ( 1.3*feat["news_7d_z_pm"].fillna(0.0)
                            + 0.3*feat["news_7d_delta_pct"].fillna(0.0)
                            - 0.6*feat["LPI_z"].fillna(0.0)
                            - 0.6*feat["WGI_MEAN_z"].fillna(0.0))

# weak labels for ranking
weak_q = 0.85
thr = float(np.quantile(feat["composite_risk_v1"], weak_q))
feat["weak_label"] = (feat["composite_risk_v1"] >= thr).astype(int)

X_cols = ["news_7d_pm","news_7d_z_pm","news_7d_delta_pct","cp_score","LPI","WGI_MEAN"]
X = feat[X_cols].copy().fillna(feat[X_cols].median(numeric_only=True))
y = feat["weak_label"].values
n = len(X)

# ----------------------- classifiers (in-sample fit) -----------------------
# RF
rf = RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample", random_state=42)
rf.fit(X, y)
feat["rf_proba"] = rf.predict_proba(X)[:,1]

# LightGBM
lgbm = lgb.LGBMClassifier(
    objective="binary", n_estimators=600, learning_rate=0.05,
    num_leaves=min(31, max(2, n-1)),
    min_data_in_leaf=1, min_sum_hessian_in_leaf=1e-3,
    feature_fraction=1.0, bagging_fraction=1.0, reg_lambda=0.1,
    random_state=42
)
lgbm.fit(X, y)
feat["lgbm_proba"] = lgbm.predict_proba(X)[:,1]

# XGBoost (CPU) — optional if libgomp/OpenMP missing on Colab
if xgb is not None:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=600, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        objective="binary:logistic", tree_method="hist", predictor="cpu_predictor",
        random_state=42
    )
    xgb_clf.fit(X, y)
    feat["xgb_proba"] = xgb_clf.predict_proba(X)[:,1]
else:
    feat["xgb_proba"] = np.nan  # placeholder so downstream columns exist

# CatBoost (CPU, silent) — optional if numpy ABI mismatch on Colab
if CatBoostClassifier is not None:
    cat = CatBoostClassifier(
        iterations=800, depth=4, learning_rate=0.05, loss_function="Logloss",
        random_seed=42, verbose=False, allow_writing_files=False, task_type="CPU"
    )
    cat.fit(X, y)
    feat["cat_proba"] = cat.predict_proba(X)[:,1]
else:
    feat["cat_proba"] = np.nan  # placeholder so downstream columns exist

# Stacked (simple average, in-sample)
clf_cols = ["rf_proba", "lgbm_proba"] + (["xgb_proba"] if xgb is not None else []) + (["cat_proba"] if CatBoostClassifier is not None else [])
feat["proba_ens"] = feat[[c for c in clf_cols if c in feat.columns]].mean(axis=1)

# ----------------------- anomalies -----------------------
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

iso = IsolationForest(n_estimators=400, contamination=0.15, random_state=42).fit(Xs)
feat["if_anom"] = -iso.score_samples(Xs)

ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.15).fit(Xs)
feat["ocsvm_anom"] = -ocsvm.score_samples(Xs)

lof = LocalOutlierFactor(n_neighbors=min(10, max(5, n-1)), contamination=0.15)
_ = lof.fit_predict(Xs)
feat["lof_anom"] = -lof.negative_outlier_factor_

# HDBSCAN outlier scores (robust to version differences); skip if import failed (e.g. numpy ABI on Colab)
try:
    if hdbscan is None:
        feat["hdb_outlier"] = 0.0
    else:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=max(3, n // 5),
            min_samples=1,
            prediction_data=True
        )
        _labels = clusterer.fit_predict(Xs)
        outlier_scores = None
        if hasattr(hdbscan, "outlier_scores"):
            outlier_scores = hdbscan.outlier_scores(clusterer)
        if outlier_scores is None or np.asarray(outlier_scores).size != len(Xs):
            soft = hdbscan.all_points_membership_vectors(clusterer)
            if soft is not None and soft.shape[0] == len(Xs):
                outlier_scores = 1.0 - soft.max(axis=1)
            else:
                outlier_scores = np.zeros(len(Xs), dtype=float)
        feat["hdb_outlier"] = pd.Series(outlier_scores, index=feat.index).fillna(0.0)
except Exception:
    feat["hdb_outlier"] = 0.0

# anomaly ensemble (includes HDBSCAN channel)
anom_parts = [feat["if_anom"], feat["ocsvm_anom"], feat["lof_anom"], feat["hdb_outlier"]]
feat["anom_ens"] = np.mean(np.vstack([p.values for p in anom_parts]), axis=0)

# new composite using ensemble proba + changepoints + structure
feat["composite_risk_v2"] = ( 1.0 * z(feat["news_7d_z_pm"].fillna(0.0))
                              + 0.25 * feat["proba_ens"]
                              + 0.15 * z(feat["cp_score"].fillna(0.0))
                              - 0.4  * feat["LPI_z"].fillna(0.0)
                              - 0.4  * feat["WGI_MEAN_z"].fillna(0.0)
                              + 0.15 * z(feat["anom_ens"].fillna(0.0)) )

# ----------------------- metrics on weak labels (in-sample) -----------------------
def safe_metric(fn, y_true, y_score):
    try:
        return float(fn(y_true, y_score))
    except Exception:
        return float("nan")

metrics = {}
_metric_names = ["rf_proba", "lgbm_proba"] + (["xgb_proba"] if xgb is not None else []) + (["cat_proba"] if CatBoostClassifier is not None else []) + ["proba_ens"]
for name in _metric_names:
    if name not in feat.columns:
        continue
    pr  = safe_metric(average_precision_score, y, feat[name].values)
    roc = safe_metric(roc_auc_score, y, feat[name].values)
    b   = safe_metric(brier_score_loss, y, feat[name].values)
    metrics[name] = {"pr_auc": pr, "roc_auc": roc, "brier": b}

# profit-optimal alert threshold on in-sample ensemble proba (proxy economics)
Cost_false_alert   = 10.0
Cost_missed_event  = 2000.0
Benefit_caught     = 800.0
proba = feat["proba_ens"].values
grid = np.linspace(0.05, 0.95, 19)
best_thr, best_profit = None, -1e9
for t in grid:
    yhat = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    profit = tp*Benefit_caught - fp*Cost_false_alert - fn*Cost_missed_event
    if profit > best_profit:
        best_profit, best_thr = profit, float(t)

# ----------------------- OOF (out-of-fold) evaluation + importance -----------------------
RNG = 42
cv = StratifiedKFold(n_splits=min(5, max(3, sum(y))), shuffle=True, random_state=RNG)

# Slightly regularized clones for OOF to reduce tiny-N overfit
rf_oof  = RandomForestClassifier(n_estimators=400, max_depth=4, min_samples_leaf=2,
                                 class_weight="balanced_subsample", random_state=RNG)
lgbm_oof = lgb.LGBMClassifier(objective="binary", n_estimators=400, learning_rate=0.05,
                              num_leaves=min(15, max(2, n-1)), min_data_in_leaf=2,
                              reg_lambda=0.5, feature_fraction=0.9, bagging_fraction=0.9,
                              random_state=RNG)
if xgb is not None:
    xgb_oof = xgb.XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                                objective="binary:logistic", tree_method="hist",
                                predictor="cpu_predictor", random_state=RNG)
if CatBoostClassifier is not None:
    cat_oof = CatBoostClassifier(iterations=600, depth=3, learning_rate=0.05,
                                 l2_leaf_reg=6.0, loss_function="Logloss",
                                 random_seed=RNG, verbose=False, allow_writing_files=False)

models_cv = {"rf": rf_oof, "lgbm": lgbm_oof}
if xgb is not None:
    models_cv["xgb"] = xgb_oof
if CatBoostClassifier is not None:
    models_cv["cat"] = cat_oof
oof = {k: np.zeros(len(X), dtype=float) for k in models_cv}

for tr, te in cv.split(X, y):
    Xtr, Xte, ytr = X.iloc[tr], X.iloc[te], y[tr]
    for name, mdl in models_cv.items():
        m = mdl.__class__(**mdl.get_params())
        m.fit(Xtr, ytr)
        oof[name][te] = m.predict_proba(Xte)[:,1]

# OOF ensemble = mean of model OOF probs (use only available models)
_oof_keys = list(models_cv.keys())
oof["ens"] = np.mean(np.vstack([oof[k] for k in _oof_keys]), axis=0)
feat["oof_ens_proba"] = oof["ens"]

# OOF metrics (realistic)
metrics_oof = {}
for name, pred in oof.items():
    metrics_oof[name] = {
        "pr_auc":  safe_metric(average_precision_score, y, pred),
        "roc_auc": safe_metric(roc_auc_score, y, pred),
        "brier":   safe_metric(brier_score_loss, y, pred)
    }

# OOF profit threshold
best_thr_oof, best_profit_oof = None, -1e9
for t in grid:
    yhat = (oof["ens"] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    profit = tp*Benefit_caught - fp*Cost_false_alert - fn*Cost_missed_event
    if profit > best_profit_oof:
        best_profit_oof, best_thr_oof = profit, float(t)

# Permutation importance on LGBM_OOF retrained on all data
lgbm_oof.fit(X, y)
perm = permutation_importance(lgbm_oof, X, y, n_repeats=200, random_state=RNG, scoring="average_precision")
imp = (pd.DataFrame({"feature": X.columns, "mean_importance": perm.importances_mean,
                     "std": perm.importances_std})
       .sort_values("mean_importance", ascending=False))
imp.to_csv(ART/"permutation_importance_lgbm.csv", index=False)

# ----------------------- outputs -----------------------
risk = feat.sort_values("composite_risk_v2", ascending=False).reset_index(drop=True)
anom = feat.sort_values("anom_ens", ascending=False).reset_index(drop=True)

risk.to_csv(GOLD/"country_risk_composite_live.csv", index=False)
anom[["iso3","anom_ens","if_anom","ocsvm_anom","lof_anom","hdb_outlier"]].to_csv(GOLD/"country_risk_anomaly_live.csv", index=False)

zT = 1.0
mu, sd = risk["composite_risk_v2"].mean(), (risk["composite_risk_v2"].std() or 1.0)
cut = mu + zT*sd
alerts = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "threshold_prob_in_sample": best_thr,
    "profit_proxy_in_sample": float(best_profit),
    "threshold_prob_oof": best_thr_oof,
    "profit_proxy_oof": float(best_profit_oof),
    "composite_field": "composite_risk_v2",
    "composite_cut_value": float(cut),
    "hits": risk[risk["composite_risk_v2"]>=cut].head(20).to_dict("records"),
    "anomaly_field": "anom_ens",
    "anomaly_top10": anom.head(10).to_dict("records"),
}
(GOLD/"alerts.json").write_text(json.dumps(alerts, indent=2))

report_in = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "models_in_sample": metrics,
    "features_used": X_cols,
    "notes": {
        "weak_labels": f"top {int((1-weak_q)*100)}% by composite_v1 used as positive class",
        "composite_v2": "mix of z(news spike), ensemble proba, change-points, governance/logistics, anomaly ensemble",
        "economics": {"Cost_false_alert": Cost_false_alert, "Cost_missed_event": Cost_missed_event, "Benefit_caught": Benefit_caught}
    }
}
(ART/"model_report.json").write_text(json.dumps(report_in, indent=2))

report_oof = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "oof_metrics": metrics_oof,
    "oof_profit_threshold": {"threshold": best_thr_oof, "proxy_profit": best_profit_oof},
    "features_used": X_cols
}
(ART/"model_report_oof.json").write_text(json.dumps(report_oof, indent=2))

print("Files written successfully:")
print(" -", (GOLD/"country_risk_composite_live.csv").as_posix())
print(" -", (GOLD/"country_risk_anomaly_live.csv").as_posix())
print(" -", (GOLD/"alerts.json").as_posix())
print(" -", (ART/"model_report.json").as_posix())
print(" -", (ART/"model_report_oof.json").as_posix())
print(" -", (ART/"permutation_importance_lgbm.csv").as_posix())

# ----------------------- LkSG register -----------------------
Z_HIGH = 1.0
Z_MED  = 0.3
r2 = risk.copy()
mu2, sd2 = r2["composite_risk_v2"].mean(), (r2["composite_risk_v2"].std() or 1.0)
r2["z"] = (r2["composite_risk_v2"] - mu2) / sd2
def action_from_z(z):
    if z >= Z_HIGH: return "Enhanced Due Diligence"
    if z >= Z_MED:  return "Monitor & Mitigate"
    return "Baseline Due Diligence"
def rationale(row):
    bits=[]
    if row.get("news_7d_z_pm",0)>0.8: bits.append("7d incident spike")
    if row.get("news_7d_delta_pct",0)>0.5: bits.append("volume > baseline")
    if pd.notna(row.get("LPI",np.nan)) and row["LPI"]<3.2: bits.append("weak logistics")
    if pd.notna(row.get("WGI_MEAN",np.nan)) and row["WGI_MEAN"]<0: bits.append("governance")
    if pd.notna(row.get("cp_score",np.nan)) and row["cp_score"]>0.8: bits.append("recent change-point")
    return ", ".join(bits) if bits else "balanced"
r2["Action"] = r2["z"].apply(action_from_z)
r2["Rationale"] = r2.apply(rationale, axis=1)
PLAYBOOK = {
    "Enhanced Due Diligence": "Escalate questionnaire; review tier-2; require CAP; increase cadence.",
    "Monitor & Mitigate":     "Track weekly; request clarifications; add clauses; prep CAP if worsens.",
    "Baseline Due Diligence": "Standard checks; periodic monitoring."
}
r2["Suggested_Next_Step"] = r2["Action"].map(PLAYBOOK)
r2.to_csv(GOLD/"lkSG_risk_register.csv", index=False)
print("LkSG register written at:", (GOLD/'lkSG_risk_register.csv').as_posix())

# ----------------------- Plotly viz -----------------------
risk["iso3"] = risk["iso3"].astype(str)
fig_map = px.choropleth(
    risk, locations="iso3", color="composite_risk_v2",
    color_continuous_scale="Reds", locationmode="ISO-3",
    hover_data={"iso3":True,"composite_risk_v2":":.2f","LPI":":.2f","WGI_MEAN":":.2f"},
    title="Composite Supply-Chain Risk v2 (live, higher = riskier)"
)
fig_map.update_layout(height=520)
fig_map.show()

topn = anom[["iso3","anom_ens"]].head(12).copy()
topn["anom_ens"] = topn["anom_ens"].astype(float)
fig_bar = px.bar(topn, x="iso3", y="anom_ens", title="Top 12 anomaly scores (ensemble)")
fig_bar.update_layout(height=360)
fig_bar.show()

# Quick peek at metrics
from pprint import pprint
print("\nWeak-label metrics (in-sample; higher PR/ROC, lower Brier is better):")
from pprint import pprint as _pprint
_pprint(metrics)
print(f"\nIn-sample profit-optimal probability threshold: {best_thr:.2f} (proxy profit={best_profit:.1f})")

print("\nOut-of-fold metrics (more realistic estimate):")
_pprint(metrics_oof)
print(f"\nOOF profit-optimal probability threshold: {best_thr_oof:.2f} (proxy profit={best_profit_oof:.1f})")

# ===========================
# BI and Insights add-on (run after the main pipeline)
# ===========================

# ---- Load outputs from your main cell (paths reused from top) ----
risk_path   = GOLD/"country_risk_composite_live.csv"
anom_path   = GOLD/"country_risk_anomaly_live.csv"
counts_pm_p = SILVER/"gdelt_counts_pm_30d.parquet"

risk = pd.read_csv(risk_path)
anom = pd.read_csv(anom_path)
risk["iso3"] = risk["iso3"].astype(str)

# Graceful presence checks
def col(c): return c in risk.columns
# Expected columns from your pipeline (use defaults if missing)
for c, default in [
    ("composite_risk_v2", np.nan), ("LPI", np.nan), ("WGI_MEAN", np.nan),
    ("news_7d_pm", 0.0), ("news_7d_z_pm", 0.0), ("news_7d_delta_pct", 0.0),
    ("cp_score", 0.0), ("proba_ens", 0.0), ("anom_ens", 0.0)
]:
    if c not in risk.columns:
        risk[c] = default

# ---- KPIs & rank tables ----
risk_sorted    = risk.sort_values("composite_risk_v2", ascending=False).reset_index(drop=True)
anom_sorted    = risk.sort_values("anom_ens", ascending=False).reset_index(drop=True)
cp_sorted      = risk.sort_values("cp_score", ascending=False).reset_index(drop=True)
topN = 10

kpis = {
    "generated_at": pd.Timestamp.utcnow().isoformat()+"Z",
    "portfolio": {
        "avg_composite_risk": float(np.nanmean(risk["composite_risk_v2"])),
        "avg_proba_ens": float(np.nanmean(risk["proba_ens"])),
        "avg_anomaly": float(np.nanmean(risk["anom_ens"])),
        "n_countries": int(risk.shape[0]),
    },
    "top_risk": risk_sorted[["iso3","composite_risk_v2","proba_ens","cp_score","anom_ens"]].head(topN).to_dict("records"),
    "top_anomaly": anom_sorted[["iso3","anom_ens","proba_ens","composite_risk_v2"]].head(topN).to_dict("records"),
    "top_movers_cp": cp_sorted[["iso3","cp_score","news_7d_z_pm","composite_risk_v2"]].head(topN).to_dict("records"),
}
(ART/"bi_kpis.json").write_text(json.dumps(kpis, indent=2))

# A compact BI table for downstream tools
bi_cols = [c for c in [
    "iso3","composite_risk_v2","proba_ens","anom_ens","cp_score",
    "news_7d_pm","news_7d_z_pm","news_7d_delta_pct","LPI","WGI_MEAN"
] if c in risk.columns]
risk[bi_cols].to_csv(ART/"bi_table.csv", index=False)

# ---- Visual 1: Top-N composite risk bar ----
fig_top = px.bar(
    risk_sorted.head(topN),
    x="iso3", y="composite_risk_v2",
    hover_data={"proba_ens":":.3f","anom_ens":":.3f","cp_score":":.2f"},
    title=f"Top {topN} Composite Risk"
)
fig_top.update_layout(height=420)
fig_top.write_html(ART/"bi_top_risk.html")

# ---- Visual 2: Risk × Anomaly quadrant ----
fig_quad = px.scatter(
    risk, x="composite_risk_v2", y="anom_ens", text="iso3",
    hover_data={"proba_ens":":.3f","cp_score":":.2f","news_7d_z_pm":":.2f"},
    title="Quadrant: Composite Risk vs Anomaly"
)
fig_quad.update_traces(mode="markers+text", textposition="top center")
# draw medians
xr, yr = np.nanmedian(risk["composite_risk_v2"]), np.nanmedian(risk["anom_ens"])
fig_quad.add_vline(x=xr, line_dash="dash", opacity=0.4)
fig_quad.add_hline(y=yr, line_dash="dash", opacity=0.4)
fig_quad.update_layout(height=520)
fig_quad.write_html(ART/"bi_quadrant_risk_anomaly.html")

# ---- Visual 3: Feature heatmap (standardized) ----
heat_cols = [c for c in ["news_7d_z_pm","news_7d_delta_pct","cp_score","LPI","WGI_MEAN","proba_ens","anom_ens"] if c in risk.columns]
heat = risk[["iso3"]+heat_cols].set_index("iso3").copy()
heat_z = (heat - heat.mean()) / (heat.std().replace(0,1))
fig_heat = px.imshow(
    heat_z, aspect="auto", color_continuous_scale="RdBu_r",
    title="Feature Heatmap (z-scored across countries)"
)
fig_heat.update_layout(height=620)
fig_heat.write_html(ART/"bi_feature_heatmap.html")

# ---- Visual 4: News intensity sparklines (last 30 days, per-million) ----
spark_ok = counts_pm_p.exists()
if spark_ok:
    mat_pm = pd.read_parquet(counts_pm_p)     # index: dates, columns: iso3
    # reshape to long for plotly
    s_long = mat_pm.reset_index(names="date").melt(id_vars="date", var_name="iso3", value_name="count_pm")
    # keep only countries we scored
    s_long = s_long[s_long["iso3"].isin(risk["iso3"])]
    # build small multiples
    fig_spark = px.line(
        s_long, x="date", y="count_pm", facet_col="iso3", facet_col_wrap=6,
        title="News Mentions per-Million (last 30 days)"
    )
    fig_spark.update_layout(height=900, showlegend=False)
    fig_spark.write_html(ART/"bi_news_sparklines.html")

# ---- Country-level inferences (drivers) ----
# Decompose composite_risk_v2 into contributions (aligned with your formula)
def explain_row(row):
    parts = {}
    parts["news_spike_z"]   = float((row.get("news_7d_z_pm",0) - risk["news_7d_z_pm"].mean()) /
                                    (risk["news_7d_z_pm"].std() or 1.0))
    parts["cp_z"]           = float((row.get("cp_score",0) - risk["cp_score"].mean()) /
                                    (risk["cp_score"].std() or 1.0))
    parts["anom_z"]         = float((row.get("anom_ens",0) - risk["anom_ens"].mean()) /
                                    (risk["anom_ens"].std() or 1.0))
    parts["LPI_z"]          = float((row.get("LPI",np.nan) - risk["LPI"].mean()) /
                                    (risk["LPI"].std() or 1.0)) if pd.notna(row.get("LPI",np.nan)) else 0.0
    parts["WGI_MEAN_z"]     = float((row.get("WGI_MEAN",np.nan) - risk["WGI_MEAN"].mean()) /
                                    (risk["WGI_MEAN"].std() or 1.0)) if pd.notna(row.get("WGI_MEAN",np.nan)) else 0.0
    # weights from your composite_v2
    contrib = {
        "Incident spike (z)":         1.0  * parts["news_spike_z"],
        "Change-point (z)":           0.15 * parts["cp_z"],
        "Anomaly ensemble (z)":       0.15 * parts["anom_z"],
        "Governance (-)":            -0.40 * parts["WGI_MEAN_z"],
        "Logistics (-)":             -0.40 * parts["LPI_z"],
        "Model ensemble (proba)":     0.25 * float(row.get("proba_ens",0))
    }
    # Top 3 drivers by absolute impact
    top3 = sorted(contrib.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    return [{"factor": k, "impact": float(v)} for k,v in top3]

insights = []
for _, r in risk_sorted.head(12).iterrows():
    insights.append({
        "iso3": r["iso3"],
        "composite_risk_v2": float(r["composite_risk_v2"]),
        "proba_ens": float(r.get("proba_ens", np.nan)),
        "anom_ens": float(r.get("anom_ens", np.nan)),
        "cp_score": float(r.get("cp_score", np.nan)),
        "top_drivers": explain_row(r)
    })
(ART/"bi_top12_drivers.json").write_text(json.dumps(insights, indent=2))

# ---- Print an executive summary in the notebook ----
def pct(x):
    try: return f"{100*x:.1f}%"
    except: return "—"

print("==== Executive Summary ====")
print(f"Portfolio size: {kpis['portfolio']['n_countries']}")
print(f"Avg composite risk: {kpis['portfolio']['avg_composite_risk']:.3f} | "
      f"Avg model proba: {kpis['portfolio']['avg_proba_ens']:.3f} | "
      f"Avg anomaly: {kpis['portfolio']['avg_anomaly']:.3f}\n")

print("Top risk countries:")
for r in kpis["top_risk"][:5]:
    print(f" - {r['iso3']}: risk={r['composite_risk_v2']:.2f}, "
          f"proba={r['proba_ens']:.2f}, anom={r['anom_ens']:.2f}, cp={r['cp_score']:.2f}")

print("\nFastest movers (change-point):")
for r in kpis["top_movers_cp"][:5]:
    print(f" - {r['iso3']}: cp={r['cp_score']:.2f}, news_z={r['news_7d_z_pm']:.2f}, risk={r['composite_risk_v2']:.2f}")

print("\nKey drivers for top 5:")
for d in insights[:5]:
    drivers = "; ".join([f"{x['factor']} ({x['impact']:+.2f})" for x in d["top_drivers"]])
    print(f" - {d['iso3']}: {drivers}")

print("\nSaved BI artifacts:")
print("  • KPIs:                 ", (ART/'bi_kpis.json').as_posix())
print("  • BI table (CSV):       ", (ART/'bi_table.csv').as_posix())
print("  • Top risk bar (HTML):  ", (ART/'bi_top_risk.html').as_posix())
print("  • Risk×Anomaly (HTML):  ", (ART/'bi_quadrant_risk_anomaly.html').as_posix())
print("  • Heatmap (HTML):       ", (ART/'bi_feature_heatmap.html').as_posix())
if spark_ok:
    print("  • Sparklines (HTML):    ", (ART/'bi_news_sparklines.html').as_posix())
print("  • Top-12 drivers JSON:  ", (ART/'bi_top12_drivers.json').as_posix())

# ===========================
# BU-level BI pack (enhanced and fixed)
# ===========================

# ----------------------------
# 0) Load inputs (risk + supplier master)
# ----------------------------
risk_p   = GOLD/"country_risk_composite_live.csv"
anom_p   = GOLD/"country_risk_anomaly_live.csv"
counts_p = SILVER/"gdelt_counts_pm_30d.parquet"
sup_p    = INT/"supplier_master.csv"

risk = pd.read_csv(risk_p) if risk_p.exists() else pd.DataFrame()
anom = pd.read_csv(anom_p) if anom_p.exists() else pd.DataFrame()

# Ensure all expected columns
for c, default in [
    ("iso3",""), ("composite_risk_v2", np.nan), ("proba_ens", np.nan), ("anom_ens", np.nan),
    ("cp_score", np.nan), ("LPI", np.nan), ("WGI_MEAN", np.nan), ("news_7d_z_pm", 0.0), ("news_7d_delta_pct", 0.0)
]:
    if c not in risk.columns: risk[c] = default
risk["iso3"] = risk["iso3"].astype(str)

# Create template supplier_master if missing
if not sup_p.exists():
    tmpl = pd.DataFrame({
        "supplier_id":[f"S{i:03d}" for i in range(1,11)],
        "supplier_name":[f"Supplier_{i:02d}" for i in range(1,11)],
        "iso3":["MEX","BRA","TUR","VNM","IND","DEU","FRA","USA","CHN","POL"],
        "business_unit":["Logistics","Manufacturing","Procurement","Manufacturing","Procurement",
                         "Logistics","Compliance","Procurement","Manufacturing","Logistics"],
        "spend":[2_000_000,1_200_000,850_000,650_000,400_000,350_000,300_000,250_000,200_000,150_000],
        "criticality":["High","High","Medium","High","Medium","Low","Medium","Low","High","Low"]
    })
    tmpl.to_csv(sup_p, index=False)
    print(f"Created template supplier_master at: {sup_p}")

sup = pd.read_csv(sup_p)
for c in ["supplier_id","supplier_name","iso3","business_unit","criticality"]:
    if c not in sup.columns: sup[c] = ""
if "spend" not in sup.columns: sup["spend"] = 0.0
sup["iso3"] = sup["iso3"].astype(str)

# ----------------------------
# 1) Supplier-level merge
# ----------------------------
merged = sup.merge(risk, on="iso3", how="left", suffixes=("",""))
for c in ["composite_risk_v2","proba_ens","anom_ens","cp_score","LPI","WGI_MEAN"]:
    if c not in merged.columns: merged[c] = np.nan

def risk_bucket(x):
    if pd.isna(x): return "Unknown"
    if x >= 1.0: return "Severe"
    if x >= 0.6: return "High"
    if x >= 0.3: return "Moderate"
    if x >= 0.1: return "Low"
    return "Very Low"
merged["risk_bucket"] = merged["composite_risk_v2"].apply(risk_bucket)
merged["spend_at_risk"] = merged["spend"] * merged["proba_ens"].fillna(0)
merged.to_csv(BUART/"suppliers_with_risk.csv", index=False)

# ----------------------------
# 2) BU Rollups
# ----------------------------
def wavg(series, weights):
    s, w = series.astype(float), weights.fillna(0.0).astype(float)
    return (s*w).sum()/w.sum() if w.sum()>0 else np.nan

agg = (merged.groupby("business_unit")
       .agg(
           suppliers=("supplier_id","nunique"),
           countries=("iso3","nunique"),
           spend_total=("spend","sum"),
           avg_risk=("composite_risk_v2","mean"),
           avg_proba=("proba_ens","mean"),
           avg_anom=("anom_ens","mean"),
           severe_cnt=("risk_bucket", lambda s: (s=="Severe").sum()),
           high_cnt=("risk_bucket", lambda s: (s=="High").sum()),
           moderate_cnt=("risk_bucket", lambda s: (s=="Moderate").sum()),
           low_cnt=("risk_bucket", lambda s: (s=="Low").sum()),
           verylow_cnt=("risk_bucket", lambda s: (s=="Very Low").sum()),
           spend_at_risk=("spend_at_risk","sum")
       )
       .reset_index()
      )

waggs = merged.groupby("business_unit").apply(
    lambda g: pd.Series({
        "wavg_risk": wavg(g["composite_risk_v2"], g["spend"]),
        "wavg_proba": wavg(g["proba_ens"], g["spend"]),
        "wavg_anom": wavg(g["anom_ens"], g["spend"])
    })
).reset_index()

bu = agg.merge(waggs, on="business_unit", how="left")
bu["risk_flag"] = np.where(bu["wavg_risk"]>=0.3,"High attention","Stable")
bu.to_csv(BUART/"bu_table.csv",index=False)

# ----------------------------
# 3) Visual BI Dashboards
# ----------------------------
fig_bu = px.bar(bu, x="business_unit", y="wavg_risk", color="risk_flag",
                title="Spend-weighted Composite Risk by BU",
                hover_data=["suppliers","countries","spend_total","wavg_proba","wavg_anom"])
fig_bu.update_layout(height=400)
fig_bu.write_html(BUART/"bu_wavg_risk.html")

fig_bubble = px.scatter(bu, x="wavg_risk", y="wavg_anom", size="spend_total",
                        color="risk_flag", hover_name="business_unit",
                        title="BU Exposure (Spend vs Risk & Anomaly)")
fig_bubble.update_layout(height=500)
fig_bubble.write_html(BUART/"bu_exposure_bubble.html")

# ----------------------------
# 4) BU Driver Decomposition
# ----------------------------
def zscore(x): return (x-x.mean())/(x.std() if x.std()>0 else 1)
for c in ["news_7d_z_pm","cp_score","anom_ens","LPI","WGI_MEAN"]:
    if c not in merged.columns: merged[c] = np.nan
merged["_z_news"]=zscore(merged["news_7d_z_pm"])
merged["_z_cp"]=zscore(merged["cp_score"])
merged["_z_anom"]=zscore(merged["anom_ens"])
merged["_z_LPI"]=zscore(merged["LPI"])
merged["_z_WGI"]=zscore(merged["WGI_MEAN"])

def contrib(r):
    return {
        "Incident spike (z)": r["_z_news"],
        "Change-point (z)": 0.15*r["_z_cp"],
        "Anomaly (z)": 0.15*r["_z_anom"],
        "Governance (-)": -0.4*r["_z_WGI"],
        "Logistics (-)": -0.4*r["_z_LPI"],
        "Model proba": 0.25*r["proba_ens"]
    }

driver_out=[]
for bu_name,g in merged.groupby("business_unit"):
    w=g["spend"].fillna(0.0)
    if w.sum()==0: w=pd.Series(np.ones(len(g)),index=g.index)
    parts=[contrib(r) for _,r in g.iterrows()]
    d=pd.DataFrame(parts)
    weighted=(d.mul(w.values[:,None])).sum()/w.sum()
    weighted=weighted.sort_values(key=lambda s:s.abs(),ascending=False)
    driver_out.append({"business_unit":bu_name,**weighted.to_dict()})
drv=pd.DataFrame(driver_out).fillna(0)
drv.to_csv(BUART/"bu_driver_contributions.csv",index=False)

# ----------------------------
# 5) Safe Merge + Scorecards with Mini BI
# ----------------------------
cols_for_merge=[c for c in ["business_unit","suppliers","countries",
                            "severe_cnt","high_cnt","moderate_cnt","low_cnt","verylow_cnt"]
                if c in agg.columns]
safe_merge=bu.merge(agg[cols_for_merge],on="business_unit",how="left")

for _,r in safe_merge.iterrows():
    bu_name=r["business_unit"]
    top_table=(merged[merged["business_unit"]==bu_name]
               .sort_values("composite_risk_v2",ascending=False)
               [["supplier_id","supplier_name","iso3","criticality","spend",
                 "composite_risk_v2","proba_ens","anom_ens","risk_bucket"]]
               .head(10).copy())
    top_table["spend"]=top_table["spend"].map(lambda v:f"{v:,.0f}")
    tbl_html=top_table.to_html(index=False,border=1)

    # Small risk gauge (plotly)
    gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=r.get("wavg_risk",0.0),
        title={'text':"Avg Risk"},
        gauge={'axis':{'range':[0,1]},
               'bar':{'color':"crimson" if r.get("wavg_risk",0)<0.3 else "orange"},
               'steps':[{'range':[0,0.3],'color':"#cce5ff"},
                        {'range':[0.3,0.6],'color':"#ffe5b4"},
                        {'range':[0.6,1.0],'color':"#f5b7b1"}]}))
    gauge_path=BUART/f"gauge_{bu_name}.html"
    gauge.write_html(gauge_path, include_plotlyjs='cdn', full_html=False)

    # Top 5 driver bar
    if bu_name in drv["business_unit"].values:
        d=drv[drv["business_unit"]==bu_name].drop(columns="business_unit").T
        d.columns=["impact"]
        d=d.sort_values("impact",key=lambda s:s.abs(),ascending=False).head(5)
        driver_fig=px.bar(d,x=d.index,y="impact",color="impact",title="Top 5 Drivers")
        driver_path=BUART/f"drivers_{bu_name}.html"
        driver_fig.write_html(driver_path, include_plotlyjs='cdn', full_html=False)
    else:
        driver_path=""

    html=f"""
    <html><head><meta charset='utf-8'><title>{bu_name} Scorecard</title></head>
    <body style='font-family:Arial;margin:20px;'>
    <h2>Business Unit Scorecard — {bu_name}</h2>
    <p><b>Generated:</b> {datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}</p>
    <h3>KPIs</h3>
    <ul>
      <li>Suppliers: {int(r.get('suppliers',len(top_table)))}</li>
      <li>Countries: {int(r.get('countries',top_table['iso3'].nunique()))}</li>
      <li>Spend total: {r.get('spend_total',0):,.0f}</li>
      <li>Spend-weighted risk: {r.get('wavg_risk',0):.2f}</li>
      <li>Spend-weighted anomaly: {r.get('wavg_anom',0):.2f}</li>
    </ul>
    <iframe src='../gauge_{bu_name}.html' width='400' height='250' frameborder='0'></iframe>
    <iframe src='../drivers_{bu_name}.html' width='500' height='300' frameborder='0'></iframe>
    <h3>Top Suppliers</h3>
    {tbl_html}
    <p><i>Charts and detailed visuals saved alongside this file.</i></p>
    </body></html>
    """
    (SCARDS/f"scorecard_{bu_name}.html").write_text(html,encoding="utf-8")

# ----------------------------
# 6) Exports Summary
# ----------------------------
print("==== BU BI Pack — Outputs ====")
print("• BU KPIs:", (BUART/'bu_table.csv'))
print("• Supplier join:", (BUART/'suppliers_with_risk.csv'))
print("• Driver contributions:", (BUART/'bu_driver_contributions.csv'))
print("• Scorecards:", SCARDS)
print("• Visuals: bar, bubble, heatmap, and mini dashboards inside /artifacts/bu/")

import shutil
import os
from google.colab import files

zip_path = '/content/live_project/artifacts/bu.zip'
directory_to_zip = '/content/live_project/artifacts/bu'

# Create the zip file
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', directory_to_zip)

# Offer the file for download
files.download(zip_path)




# Economic assumptions (you can tune)
ECON = {
    "Cost_false_alert":   10_000.0,   # analyst time + noise cost per FP
    "Cost_missed_event": 200_000.0,   # line-down/stockout $ impact per FN (conservative)
    "Benefit_caught":     80_000.0,   # avoided expediting/reroute etc per TP
    "Lead_time_days_gain": 5          # typical lead-time advantage if flagged early
}

TOPK = 5                 # top-K for backtest precision@K / recall@K
WEAK_POS_QUANT = 0.85    # weak label quantile if needed
THRESH_GRID = np.round(np.linspace(0.05, 0.95, 19), 2)


# ----------------------------1) Load current outputs----------------------------
risk_p   = GOLD/"country_risk_composite_live.csv"
alerts_p = GOLD/"alerts.json"

if not risk_p.exists():
    raise FileNotFoundError(f"Missing {risk_p}. Run main pipeline first.")

risk = pd.read_csv(risk_p)
risk["iso3"] = risk["iso3"].astype(str)
# graceful defaults
for c, default in [
    ("proba_ens", np.nan), ("composite_risk_v2", np.nan), ("news_7d_z_pm", 0.0),
    ("cp_score", 0.0), ("anom_ens", 0.0), ("LPI", np.nan), ("WGI_MEAN", np.nan)
]:
    if c not in risk.columns: risk[c] = default

alerts = {}
if alerts_p.exists():
    try:
        alerts = json.loads(alerts_p.read_text())
    except Exception:
        alerts = {}

# Weak labels (if not provided): top (1-quantile) by composite_v2
if "weak_label" not in risk.columns:
    qthr = risk["composite_risk_v2"].quantile(WEAK_POS_QUANT)
    risk["weak_label"] = (risk["composite_risk_v2"] >= qthr).astype(int)

# ----------------------------
# 2) ROI Curves & Sensitivity (“tornado”)
# ----------------------------
def confusion_from_threshold(p, y, thr):
    yhat = (p >= thr).astype(int)
    tp = int(((y==1)&(yhat==1)).sum())
    fp = int(((y==0)&(yhat==1)).sum())
    fn = int(((y==1)&(yhat==0)).sum())
    tn = int(((y==0)&(yhat==0)).sum())
    return tp, fp, fn, tn

def profit(tp, fp, fn, econ):
    return tp*econ["Benefit_caught"] - fp*econ["Cost_false_alert"] - fn*econ["Cost_missed_event"]

y = risk["weak_label"].values
p = risk["proba_ens"].fillna(0.0).values

roi = []
for thr in THRESH_GRID:
    tp, fp, fn, tn = confusion_from_threshold(p, y, thr)
    pr = profit(tp, fp, fn, ECON)
    roi.append({"threshold": float(thr), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                "profit": float(pr)})

roi_df = pd.DataFrame(roi).sort_values("threshold")
roi_df.to_csv(UPG/"roi_curve.csv", index=False)

fig_roi = px.line(roi_df, x="threshold", y="profit",
                  title="ROI vs Threshold (current snapshot)",
                  markers=True)
fig_roi.update_layout(height=400)
fig_roi.write_html(UPG/"roi_curve.html")

# Sensitivity tornado: vary each econ knob ±30%
def roi_with_econ(econ_overrides):
    econ = {**ECON, **econ_overrides}
    # pick current best thr from base case
    base_best_thr = roi_df.loc[roi_df["profit"].idxmax(), "threshold"]
    tp, fp, fn, tn = confusion_from_threshold(p, y, base_best_thr)
    return profit(tp, fp, fn, econ)

sens = []
for k in ["Benefit_caught","Cost_false_alert","Cost_missed_event"]:
    base = roi_with_econ({})
    up   = roi_with_econ({k: ECON[k]*1.3})
    dn   = roi_with_econ({k: ECON[k]*0.7})
    sens.append({"factor": k, "delta_up": float(up-base), "delta_down": float(dn-base), "base_profit": float(base)})
sens_df = pd.DataFrame(sens)
sens_df.to_csv(UPG/"roi_sensitivity_tornado.csv", index=False)

fig_tornado = go.Figure()
for _, r in sens_df.iterrows():
    fig_tornado.add_trace(go.Bar(
        x=[r["delta_up"]],
        y=[r["factor"]],
        orientation='h',
        name="+30%",
        marker_color="seagreen"
    ))
    fig_tornado.add_trace(go.Bar(
        x=[r["delta_down"]],
        y=[r["factor"]],
        orientation='h',
        name="-30%",
        marker_color="indianred"
    ))
fig_tornado.update_layout(barmode='relative', title="ROI Sensitivity (Tornado, base thr)", height=420)
fig_tornado.write_html(UPG/"roi_sensitivity_tornado.html")

# ----------------------------
# 3) Rolling Back-test (if history snapshots exist)
#    Expect structure: /content/live_project/history/YYYY-MM-DD/country_risk_composite_live.csv
# ----------------------------
def list_history():
    if not HISTORY.exists(): return []
    out=[]
    for d in sorted([p for p in HISTORY.iterdir() if p.is_dir()]):
        f = d/"country_risk_composite_live.csv"
        if f.exists(): out.append(f)
    return out

hist_files = list_history()
backtest = []
if len(hist_files) >= 3:
    # Use monthly “next-period” event = top X% risk in next snapshot
    # Predict at time t using proba_ens_t; evaluate against events at t+1
    for i in range(len(hist_files)-1):
        f_t   = hist_files[i]
        f_t1  = hist_files[i+1]
        date_t  = f_t.parent.name
        date_t1 = f_t1.parent.name
        df_t  = pd.read_csv(f_t)
        df_t1 = pd.read_csv(f_t1)
        # ensure cols
        for c in ["iso3","proba_ens","composite_risk_v2"]:
            if c not in df_t.columns: df_t[c] = np.nan
            if c not in df_t1.columns: df_t1[c] = np.nan
        df_t["iso3"] = df_t["iso3"].astype(str)
        df_t1["iso3"] = df_t1["iso3"].astype(str)

        # Define "event@t1" as top quantile of composite at t+1
        qthr = df_t1["composite_risk_v2"].quantile(WEAK_POS_QUANT)
        events = (df_t1.set_index("iso3")["composite_risk_v2"] >= qthr).astype(int)

        # predictions at t: top-K by proba_ens
        preds = df_t[["iso3","proba_ens"]].dropna().sort_values("proba_ens", ascending=False)
        topk = preds.head(TOPK)["iso3"].tolist()

        # Compute precision@K, recall@K
        y_true = events.reindex(preds["iso3"]).fillna(0).astype(int)
        k = min(TOPK, len(preds))
        precK = float(y_true.loc[topk].mean()) if k>0 else np.nan
        # recallK = (#events caught in topK) / (#events total)
        total_events = int(events.sum())
        recallK = float(y_true.loc[topk].sum()/total_events) if total_events>0 and k>0 else np.nan

        backtest.append({"t": date_t, "t_plus": date_t1, "precision@K": precK, "recall@K": recallK,
                         "K": int(TOPK), "events_total": int(total_events)})

bt_df = pd.DataFrame(backtest)
if not bt_df.empty:
    bt_df.to_csv(UPG/"rolling_backtest_prek_reck.csv", index=False)
    fig_bt = px.line(bt_df, x="t", y=["precision@K","recall@K"], markers=True,
                     title=f"Rolling Back-test (K={TOPK})")
    fig_bt.update_layout(height=420)
    fig_bt.write_html(UPG/"rolling_backtest_prek_reck.html")

# ----------------------------
# 4) Ablation (drop feature blocks from current composite)
# ----------------------------
def z_s(col):
    s = col.astype(float)
    sd = s.std()
    return (s - s.mean()) / (sd if sd and sd>0 else 1.0)

risk["_z_news"] = z_s(risk["news_7d_z_pm"].fillna(0))
risk["_z_cp"]   = z_s(risk["cp_score"].fillna(0))
risk["_z_anom"] = z_s(risk["anom_ens"].fillna(0))
risk["_z_LPI"]  = z_s(risk["LPI"].fillna(risk["LPI"].median() if pd.notna(risk["LPI"].median()) else 0))
risk["_z_WGI"]  = z_s(risk["WGI_MEAN"].fillna(risk["WGI_MEAN"].median() if pd.notna(risk["WGI_MEAN"].median()) else 0))

def composite_full(df):
    return ( 1.0*df["_z_news"] + 0.25*df["proba_ens"].fillna(0)
           + 0.15*df["_z_cp"]  - 0.4*df["_z_LPI"] - 0.4*df["_z_WGI"]
           + 0.15*df["_z_anom"] )

ablations = {}
base = composite_full(risk).values
ablations["base"] = base

ablations["drop_news"]  = (0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values
ablations["drop_cp"]    = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values
ablations["drop_anom"]  = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"]).values
ablations["drop_LPI"]   = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values
ablations["drop_WGI"]   = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] + 0.15*risk["_z_anom"]).values
ablations["drop_proba"] = (1.0*risk["_z_news"] + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values

# Evaluate delta in rank correlation vs base (Spearman)
from scipy.stats import spearmanr
rows=[]
for name, vals in ablations.items():
    rho = float(spearmanr(base, vals, nan_policy="omit").correlation)
    rows.append({"ablation": name, "spearman_to_base": rho})
abl_df = pd.DataFrame(rows).sort_values("spearman_to_base", ascending=False)
abl_df.to_csv(UPG/"ablation_rank_stability.csv", index=False)
fig_abl = px.bar(abl_df, x="ablation", y="spearman_to_base",
                 title="Ablation: Rank Stability vs Base Composite")
fig_abl.update_layout(height=420)
fig_abl.write_html(UPG/"ablation_rank_stability.html")

# Optional: ROI delta if we re-threshold on ablated score (using weak labels)
def roi_from_score(score):
    local = []
    for thr in THRESH_GRID:
        yhat = (score >= thr).astype(int)
        tp = int(((y==1)&(yhat==1)).sum())
        fp = int(((y==0)&(yhat==1)).sum())
        fn = int(((y==1)&(yhat==0)).sum())
        local.append({"thr": float(thr), "profit": profit(tp, fp, fn, ECON)})
    ldf = pd.DataFrame(local)
    return float(ldf["profit"].max())

roi_rows=[]
for name, vals in ablations.items():
    roi_rows.append({"score": name, "best_profit": roi_from_score(vals)})
pd.DataFrame(roi_rows).to_csv(UPG/"ablation_roi.csv", index=False)

# ----------------------------
# 5) Stress Tests (±10/20/30% perturbations)
# ----------------------------
def stress_score(df, pct):
    out = df.copy()
    out["_z_news_s"] = z_s(df["news_7d_z_pm"].fillna(0)*(1+pct))
    out["_z_cp_s"]   = z_s(df["cp_score"].fillna(0)*(1+pct))
    out["_z_anom_s"] = z_s(df["anom_ens"].fillna(0)*(1+pct))
    out["_z_LPI_s"]  = z_s(df["LPI"].fillna(0)*(1-pct))  # inverse for “-” terms = conservative
    out["_z_WGI_s"]  = z_s(df["WGI_MEAN"].fillna(0)*(1-pct))
    return ( 1.0*out["_z_news_s"] + 0.25*out["proba_ens"].fillna(0)
           + 0.15*out["_z_cp_s"]  - 0.4*out["_z_LPI_s"] - 0.4*out["_z_WGI_s"]
           + 0.15*out["_z_anom_s"] ).values

base_score = composite_full(risk).values
stress = []
for pct in [0.1, 0.2, 0.3]:
    s = stress_score(risk, pct)
    rho = float(spearmanr(base_score, s, nan_policy="omit").correlation)
    # stability of alert set at current best threshold
    best_thr = roi_df.loc[roi_df["profit"].idxmax(), "threshold"]
    base_set = set(risk.loc[base_score>=best_thr,"iso3"])
    stress_set = set(risk.loc[s>=best_thr,"iso3"])
    jacc = (len(base_set & stress_set) / len(base_set | stress_set)) if (base_set or stress_set) else 1.0
    stress.append({"pct": pct, "spearman_to_base": rho, "alert_jaccard": float(jacc)})
stress_df = pd.DataFrame(stress)
stress_df.to_csv(UPG/"stress_stability.csv", index=False)
fig_str = px.bar(stress_df.melt(id_vars="pct"), x="pct", y="value", color="variable",
                 title="Stress Test: Rank/Alert Set Stability", barmode="group")
fig_str.update_layout(height=420)
fig_str.write_html(UPG/"stress_stability.html")

# ----------------------------
# 6) Causality-ish & Bias Checks
# ----------------------------
# Placebo: shuffle iso3 mapping, recompute corr
shuf = risk.copy()
shuf = shuf.sample(frac=1.0, random_state=42).reset_index(drop=True)
rho_placebo = float(spearmanr(base_score, composite_full(shuf).values, nan_policy="omit").correlation)
json.dump({"placebo_spearman": rho_placebo}, open(UPG/"placebo_check.json","w"), indent=2)

# Bias stratification by structure tiers
def tier_by(col, q=[0.33, 0.66]):
    s = risk[col]
    try:
        b = s.quantile(q).values
    except Exception:
        b = [s.median(), s.median()]
    def lab(v):
        if pd.isna(v): return "Unknown"
        if v <= b[0]: return "Low"
        if v <= b[1]: return "Mid"
        return "High"
    return s.apply(lab)

risk["tier_LPI"] = tier_by("LPI")
risk["tier_WGI"] = tier_by("WGI_MEAN")

bias_rows=[]
for dim in ["tier_LPI","tier_WGI"]:
    for lvl, g in risk.groupby(dim):
        if len(g)==0: continue
        # at best_thr from base ROI
        best_thr = roi_df.loc[roi_df["profit"].idxmax(), "threshold"]
        tp, fp, fn, tn = confusion_from_threshold(g["proba_ens"].fillna(0).values,
                                                  g["weak_label"].values, best_thr)
        bias_rows.append({
            "dimension": dim, "level": str(lvl),
            "n": int(len(g)),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn
        })
bias_df = pd.DataFrame(bias_rows)
bias_df.to_csv(UPG/"bias_stratified_confusion.csv", index=False)

# ----------------------------7) Model Card & Governance----------------------------
model_card = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "objective": "Early detection of country-level supply chain risk using news-based signals + structure.",
    "sources": ["GDELT timelines (30d live)", "World Bank LPI/WGI", "Population (per-million)"],
    "models": ["RF, LightGBM, XGBoost, CatBoost (stacked)", "Anomaly ensemble (IF, OCSVM, LOF, HDBSCAN)"],
    "composite": "1.0*z(news spike) + 0.25*proba_ens + 0.15*z(cp) - 0.4*z(LPI) - 0.4*z(WGI) + 0.15*z(anom)",
    "refresh_cadence": "Daily for GDELT; annual for LPI/WGI (latest available).",
    "monitoring": {
        "drift": ["news intensity drift", "class balance", "threshold stability"],
        "alerts": ["Severe/High/Moderate buckets with playbook"]
    },
    "known_limits": [
        "Media bias/coverage bias by country",
        "Weak labels proxy true disruptions",
        "Short horizon on live stream (extend with history for strategy)"
    ],
    "econ_assumptions": ECON,
    "artifacts": {
        "roi_curve_csv": (UPG/"roi_curve.csv").as_posix(),
        "roi_curve_html": (UPG/"roi_curve.html").as_posix(),
        "roi_sensitivity_html": (UPG/"roi_sensitivity_tornado.html").as_posix(),
        "rolling_backtest_html": (UPG/"rolling_backtest_prek_reck.html").as_posix() if (UPG/"rolling_backtest_prek_reck.html").exists() else None,
        "ablation_html": (UPG/"ablation_rank_stability.html").as_posix(),
        "stress_html": (UPG/"stress_stability.html").as_posix(),
        "bias_csv": (UPG/"bias_stratified_confusion.csv").as_posix()
    }
}
(UPG/"model_card.json").write_text(json.dumps(model_card, indent=2))

# Pretty Markdown for faculty / stakeholders
mc_md = f"""# Model Card — Live Supply-Chain Risk

**Generated:** {model_card['generated_at']}

## Objective
Early detection of country-level supply chain risk to prioritize due diligence and mitigation.

## Data & Signals
- **GDELT** news timelines (live 30d; extendable)
- **World Bank** LPI & WGI (structure)
- Population normalization (per-million)

## Model Stack
- RF, LightGBM, XGBoost, CatBoost (stacked)
- Anomaly ensemble: IsolationForest, OneClassSVM, LOF, HDBSCAN
- Composite: `1.0*z(news) + 0.25*proba + 0.15*z(cp) - 0.4*z(LPI) - 0.4*z(WGI) + 0.15*z(anom)`

## Validation & Ops
- ROI curves & sensitivity tornado
- Rolling back-test (if history exists)
- Feature ablation & stress tests
- Bias stratification by LPI/WGI tiers
- Drift monitors: news intensity, class balance, threshold stability

## Known Limits
Media bias, weak labels proxy, structural data refresh lag.

## Economics (assumptions)
{json.dumps(ECON, indent=2)}

## Key Artifacts
- ROI curve: { (UPG/'roi_curve.html').as_posix() }
- Sensitivity tornado: { (UPG/'roi_sensitivity_tornado.html').as_posix() }
- Ablation: { (UPG/'ablation_rank_stability.html').as_posix() }
- Stress stability: { (UPG/'stress_stability.html').as_posix() }
- Back-test: { (UPG/'rolling_backtest_prek_reck.html').as_posix() if (UPG/'rolling_backtest_prek_reck.html').exists() else '—' }
- Bias CSV: { (UPG/'bias_stratified_confusion.csv').as_posix() }
"""
(UPG/"model_card.md").write_text(mc_md, encoding="utf-8")


# ----------------------------8) Productization Stubs (scheduler + API)----------------------------
# a) Cron/GitHub Actions style shell template
cron_sh = """#!/usr/bin/env bash
set -e
# Example: schedule this script daily with cron or GitHub Actions runner
# 1) Run main pipeline (python or colab nbconvert)
# 2) Run BI packs
# 3) Run upgrade pack
# 4) Sync artifacts to object storage (S3/GCS) or BI server

python main_pipeline.py
python bi_pack.py
python bu_bi_pack.py
python upgrade_pack.py

# rsync or aws s3 cp artifacts
# aws s3 sync /content/live_project/artifacts s3://your-bucket/live_project/artifacts --delete
"""
(UPG/"cron_template.sh").write_text(cron_sh)

# b) Minimal FastAPI stub (serves alerts & KPIs)
fastapi_stub = """from fastapi import FastAPI
from fastapi.responses import JSONResponse
import json, pathlib

BASE = pathlib.Path('/content/live_project')
GOLD = BASE/'data'/'gold'
ART  = BASE/'artifacts'
UPG  = ART/'upgrade'

app = FastAPI(title='Supply Chain Risk API', version='0.1')

def load_json(p):
    if p.exists():
        try: return json.loads(p.read_text())
        except: return {}
    return {}

@app.get('/health')
def health(): return {'ok': True}

@app.get('/alerts')
def alerts():
    return JSONResponse(load_json(GOLD/'alerts.json'))

@app.get('/kpis')
def kpis():
    return JSONResponse(load_json(ART/'bi_kpis.json'))

@app.get('/model_card')
def model_card():
    return JSONResponse(load_json(UPG/'model_card.json'))
"""
(UPG/"fastapi_stub.py").write_text(fastapi_stub)


# ----------------------------9) Console Summary----------------------------

print("==== Upgrade Pack — Artifacts ====")
print("• ROI curve CSV/HTML:", (UPG/"roi_curve.csv").as_posix(), "|", (UPG/"roi_curve.html").as_posix())
print("• ROI sensitivity (tornado):", (UPG/"roi_sensitivity_tornado.html").as_posix())
if (UPG/"rolling_backtest_prek_reck.html").exists():
    print("• Rolling back-test:", (UPG/"rolling_backtest_prek_reck.html").as_posix())
else:
    print("• Rolling back-test: (skipped — add history snapshots under /content/live_project/history/YYYY-MM-DD/)")
print("• Ablation rank stability:", (UPG/"ablation_rank_stability.html").as_posix())
print("• Stress stability:", (UPG/"stress_stability.html").as_posix())
print("• Bias stratification CSV:", (UPG/"bias_stratified_confusion.csv").as_posix())
print("• Model card JSON/MD:", (UPG/"model_card.json").as_posix(), "|", (UPG/"model_card.md").as_posix())
print("• Productization: cron_template.sh, fastapi_stub.py in", UPG.as_posix())

# Friendly hints
print("\nTips:")
print(" - To enable rolling back-test, create monthly folders like /history/2025-01-01/country_risk_composite_live.csv (and so on).")
print(" - Tune ECON in this cell to your company’s costs; the tornado will update.")
print(" - Serve alerts & KPIs: `uvicorn artifacts/upgrade/fastapi_stub:app --reload --host 0.0.0.0 --port 8000`")

# -*- coding: utf-8 -*-
"""Untitled10 (1).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/13FsNfzprNld5agTpWTrDZ28dgPZcDMKQ
"""

# ---- installs (single shot) ----
!pip -q install pandas numpy requests scikit-learn lightgbm ruptures plotly tqdm

# ---- imports & setup ----
import os, json, math, time, warnings, pathlib, re
from datetime import datetime, timedelta
from typing import List, Optional, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

import lightgbm as lgb
import ruptures as rpt
import plotly.express as px
from urllib.parse import urlparse
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)

# ---- paths ----
BASE   = pathlib.Path("/content/live_project")
BRONZE = BASE/"data"/"bronze"; BRONZE.mkdir(parents=True, exist_ok=True)
SILVER = BASE/"data"/"silver"; SILVER.mkdir(parents=True, exist_ok=True)
GOLD   = BASE/"data"/"gold"  ; GOLD.mkdir(parents=True, exist_ok=True)
ART    = BASE/"artifacts";     ART.mkdir(parents=True, exist_ok=True)

# ---- watchlist (ISO3) ----
ISO3 = ["DEU","NLD","POL","FRA","ITA","ESP","CZE","AUT","SWE","DNK","NOR","FIN",
        "GBR","IRL","USA","CAN","MEX","BRA","CHN","JPN","KOR","IND","THA","VNM","TUR"]

ISO2_TO_ISO3 = {
    "DE":"DEU","NL":"NLD","PL":"POL","FR":"FRA","IT":"ITA","ES":"ESP","CZ":"CZE","AT":"AUT","SE":"SWE",
    "DK":"DNK","NO":"NOR","FI":"FIN","GB":"GBR","UK":"GBR","IE":"IRL","US":"USA","CA":"CAN","MX":"MEX",
    "BR":"BRA","CN":"CHN","JP":"JPN","KR":"KOR","IN":"IND","TH":"THA","VN":"VNM","TR":"TUR"
}
ISO3_TO_ISO2 = {v:k for k,v in ISO2_TO_ISO3.items()}

# ---- HTTP session with retries ----
def make_session():
    s = requests.Session()
    retry = Retry(
        total=6, backoff_factor=0.6,
        status_forcelist=(429,500,502,503,504),
        allowed_methods=("GET","HEAD","OPTIONS")
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent":"lksg-risk-colab/1.0"})
    return s
SESSION = make_session()

# ---- small helpers ----
def fresh(path: pathlib.Path, ttl_minutes=15) -> bool:
    return path.exists() and (time.time() - path.stat().st_mtime) < ttl_minutes*60

def zscore(col: pd.Series) -> pd.Series:
    sd = col.std()
    return (col - col.mean()) / (sd if (sd and sd>0) else 1.0)


# ----------------------------1) WORLD BANK (batched, cached) — LPI / POP / WGI----------------------------
WB_BASE = "https://api.worldbank.org/v2/country/{ISOS}/indicator/{IND}"

def wb_fetch_indicator_batched(iso_list, indicator, mrv=1) -> pd.DataFrame:
    isos = ";".join(iso_list)
    params = {"format":"json","mrv":str(mrv),"per_page":"20000"}
    r = SESSION.get(WB_BASE.format(ISOS=isos, IND=indicator), params=params, timeout=60)
    r.raise_for_status()
    js = r.json()
    rows=[]
    if isinstance(js,list) and len(js)==2 and js[1]:
        for rec in js[1]:
            if rec and rec.get("value") is not None:
                rows.append({
                    "iso3": rec.get("countryiso3code"),
                    "date": int(rec.get("date")) if rec.get("date") else None,
                    "indicator": indicator,
                    "value": float(rec.get("value"))
                })
    return pd.DataFrame(rows)

def wb_latest(df, indicator, alias) -> pd.DataFrame:
    if df.empty: return pd.DataFrame({"iso3": ISO3, alias: np.nan})
    d = df[df["indicator"]==indicator].sort_values(["iso3","date"]).groupby("iso3").tail(1)
    out = d[["iso3","value"]].rename(columns={"value":alias})
    miss = [c for c in ISO3 if c not in set(out["iso3"])]
    if miss: out = pd.concat([out, pd.DataFrame({"iso3":miss, alias:np.nan})], ignore_index=True)
    return out

LPI = "LP.LPI.OVRL.XQ"
POP = "SP.POP.TOTL"
WGI_CODES = ["GE.EST","RQ.EST","RL.EST","CC.EST"]

def get_worldbank_data(ttl_min=30):
    lpi_p, pop_p, wgi_p = BRONZE/"wb_lpi.parquet", BRONZE/"wb_pop.parquet", BRONZE/"wb_wgi.parquet"
    if fresh(lpi_p, ttl_min): wb_lpi = pd.read_parquet(lpi_p)
    else:
        wb_lpi = wb_fetch_indicator_batched(ISO3, LPI, mrv=1)
        if not wb_lpi.empty: wb_lpi.to_parquet(lpi_p, index=False)

    if fresh(pop_p, ttl_min): wb_pop = pd.read_parquet(pop_p)
    else:
        wb_pop = wb_fetch_indicator_batched(ISO3, POP, mrv=1)
        if not wb_pop.empty: wb_pop.to_parquet(pop_p, index=False)

    if fresh(wgi_p, ttl_min): wb_wgi = pd.read_parquet(wgi_p)
    else:
        parts = [wb_fetch_indicator_batched(ISO3, c, mrv=1) for c in WGI_CODES]
        wb_wgi = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
        if not wb_wgi.empty: wb_wgi.to_parquet(wgi_p, index=False)
    return wb_lpi, wb_pop, wb_wgi

wb_lpi, wb_pop, wb_wgi = get_worldbank_data()

lpi_latest = wb_latest(wb_lpi, LPI, "LPI")
pop_latest = wb_latest(wb_pop, POP, "POP")

if not wb_wgi.empty:
    wgi = (wb_wgi.sort_values(["iso3","date"])
           .groupby(["iso3","indicator"]).tail(1)
           .pivot(index="iso3", columns="indicator", values="value").reset_index())
    for c in WGI_CODES:
        if c not in wgi.columns: wgi[c] = np.nan
    wgi["WGI_MEAN"] = wgi[WGI_CODES].mean(axis=1)
    wgi = wgi[["iso3","WGI_MEAN"]]
else:
    wgi = pd.DataFrame({"iso3": ISO3, "WGI_MEAN": np.nan})


# ----------------------------2) GDELT TIMELINE 30d → incident features (per-million)----------------------------
def gdelt_timeline_matrix(query: str, timespan="30d") -> pd.DataFrame:
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {"query": query, "mode":"timelinesourcecountry", "timespan": timespan, "format":"JSON"}
    r = SESSION.get(url, params=params, timeout=60)
    r.raise_for_status()
    js = r.json()
    tl = js.get("timeline", [])
    return pd.DataFrame(tl) if tl else pd.DataFrame()

QUERY_TIMELINE = ("(port OR strike OR flood OR storm OR customs OR wildfire OR landslide OR earthquake OR cyber) "
                  "AND (delay OR disruption OR shutdown OR congestion OR outage OR blockade)")

raw = gdelt_timeline_matrix(QUERY_TIMELINE, "30d")

def ensure_date_series(df: pd.DataFrame) -> pd.Series:
    for cand in ("date","datetime","time","timestamp"):
        if cand in df.columns:
            s = pd.to_datetime(df[cand].astype(str), errors="coerce")
            return s.dt.normalize()
    end = pd.Timestamp.utcnow().normalize()
    start = end - pd.Timedelta(days=max(len(df)-1,0))
    return pd.Series(pd.date_range(start, end, freq="D"), index=df.index)

# build count matrix (30d × ISO3)
if raw.empty:
    dates = pd.date_range(pd.Timestamp.utcnow().normalize()-pd.Timedelta(days=29),
                          pd.Timestamp.utcnow().normalize(), freq="D")
    mat = pd.DataFrame(0.0, index=dates, columns=ISO3)
else:
    raw["date"] = ensure_date_series(raw)
    iso2_cols = [c for c in raw.columns if c != "date"]
    long = raw.melt(id_vars=["date"], value_vars=iso2_cols, var_name="iso2", value_name="count").fillna(0)
    long["iso3"] = long["iso2"].str.upper().map(ISO2_TO_ISO3)
    long = long.dropna(subset=["iso3"])
    long = long[long["iso3"].isin(ISO3)]
    mat = long.pivot_table(index="date", columns="iso3", values="count", aggfunc="sum").sort_index()
    for c in ISO3:
        if c not in mat.columns: mat[c]=0.0
    end = mat.index.max() if len(mat.index) else pd.Timestamp.utcnow().normalize()
    start = end - pd.Timedelta(days=29)
    mat = mat.reindex(pd.date_range(start, end, freq="D"), fill_value=0.0)
    mat = mat[ISO3]

mat.astype(float).to_parquet(SILVER/"gdelt_counts_30d.parquet")

# per-million normalize
POP_MAP = dict(zip(pop_latest["iso3"], pop_latest["POP"])) if not pop_latest.empty else {}
divisors = pd.Series({c: (POP_MAP.get(c, np.nan)/1_000_000.0) for c in mat.columns})
if divisors.notna().any():
    fill_val = float(divisors.dropna().median())
    divisors = divisors.fillna(fill_val if not np.isnan(fill_val) else 1.0)
else:
    divisors = pd.Series(1.0, index=mat.columns)
mat_pm = mat.divide(divisors, axis=1).astype(float)
mat_pm.to_parquet(SILVER/"gdelt_counts_pm_30d.parquet")

# rolling features
pm_7d  = mat_pm.rolling(window=7,  min_periods=1).sum().iloc[-1]
pm_30m = mat_pm.rolling(window=30, min_periods=7).mean().iloc[-1]
pm_30s = mat_pm.rolling(window=30, min_periods=7).std(ddof=0).iloc[-1]
eps = 1e-6
news_feats = pd.DataFrame({
    "iso3": mat_pm.columns,
    "news_7d_pm": pm_7d.values.astype(float),
    "news_7d_z_pm": ((pm_7d - (pm_30m*7.0)) / (np.sqrt((pm_30s**2)*7.0) + eps)).values.astype(float),
    "news_7d_delta_pct": ((pm_7d - (pm_30m*7.0)) / ((pm_30m*7.0) + eps)).values.astype(float),
}).sort_values("iso3").reset_index(drop=True)
news_feats.to_csv(SILVER/"news_feats_gdelt_pm.csv", index=False)

# ----------------------------3) CHANGEPOINTS on 30d PM series (ruptures / PELT)----------------------------
def changepoint_score(series: pd.Series) -> float:
    y = series.values.astype(float)
    if np.allclose(y, y[0]) or len(y)<15:
        return 0.0
    try:
        algo = rpt.Pelt(model="rbf").fit(y)
        bkps = algo.predict(pen=1.0)
        if len(bkps) <= 1: return 0.0
        last_cp = sorted(bkps[:-1])[-1]
        pre  = y[max(0,last_cp-7): last_cp]
        post = y[last_cp: min(len(y), last_cp+7)]
        if len(pre)==0 or len(post)==0: return 0.0
        diff = np.mean(post) - np.mean(pre)
        std  = np.std(y) if np.std(y)>0 else 1.0
        return float(diff/std)
    except Exception:
        return 0.0

cp_df = pd.DataFrame({"iso3": mat_pm.columns,
                      "cp_score": [changepoint_score(mat_pm[c]) for c in mat_pm.columns]})
cp_df.to_csv(SILVER/"gdelt_changepoint_scores.csv", index=False)


# ----------------------------4) FEATURE TABLE + MODELS (RF + LGBM) + ANOMALY ENSEMBLE----------------------------
feat = (lpi_latest.merge(wgi, on="iso3", how="left")
                 .merge(pop_latest, on="iso3", how="left")
                 .merge(news_feats, on="iso3", how="left")
                 .merge(cp_df, on="iso3", how="left"))

feat["LPI_z"] = zscore(feat["LPI"])
feat["WGI_MEAN_z"] = zscore(feat["WGI_MEAN"])

# base composite for weak labels
feat["composite_risk"] = ( 1.3*feat["news_7d_z_pm"].fillna(0.0)
                         + 0.3*feat["news_7d_delta_pct"].fillna(0.0)
                         - 0.6*feat["LPI_z"].fillna(0.0)
                         - 0.6*feat["WGI_MEAN_z"].fillna(0.0) )

weak_q = 0.85
thr = float(np.quantile(feat["composite_risk"], weak_q))
feat["weak_label"] = (feat["composite_risk"] >= thr).astype(int)

X_cols = ["news_7d_pm","news_7d_z_pm","news_7d_delta_pct","cp_score","LPI","WGI_MEAN"]
X = feat[X_cols].copy().fillna(feat[X_cols].median(numeric_only=True))
y = feat["weak_label"].values
n = len(X)

# RF (calibrated)
rf_raw = RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample", random_state=42)
rf = CalibratedClassifierCV(rf_raw, method="isotonic", cv=min(3, max(2, 3)))
rf.fit(X, y)
feat["rf_proba"] = rf.predict_proba(X)[:,1]

# LightGBM (small-N friendly)
lgbm = lgb.LGBMClassifier(
    objective="binary", n_estimators=500, learning_rate=0.05,
    num_leaves=min(15, max(2, n-1)), min_data_in_leaf=1,
    min_sum_hessian_in_leaf=1e-3, feature_fraction=1.0,
    bagging_fraction=1.0, reg_lambda=0.1, random_state=42
)
lgbm.fit(X, y)
feat["lgbm_proba"] = lgbm.predict_proba(X)[:,1]

# meta blender (logistic on RF/LGBM)
P = feat[["rf_proba","lgbm_proba"]].values
meta = LogisticRegression(max_iter=1000).fit(P, y)
feat["proba_ens"] = meta.predict_proba(P)[:,1]

# anomaly ensemble: IF + OCSVM + LOF (scaled)
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

iso = IsolationForest(n_estimators=400, contamination=0.15, random_state=42).fit(Xs)
oc  = OneClassSVM(kernel="rbf", gamma="scale", nu=0.15).fit(Xs)
lof = LocalOutlierFactor(n_neighbors=min(10, max(5, n-1)), contamination=0.15)
_ = lof.fit_predict(Xs)

feat["if_anom"]    = -iso.score_samples(Xs)
feat["ocsvm_anom"] = -oc.score_samples(Xs)
feat["lof_anom"]   = -lof.negative_outlier_factor_
feat["anom_ens"]   = np.mean(np.vstack([feat["if_anom"], feat["ocsvm_anom"], feat["lof_anom"]]), axis=0)

# composite v2 (learned + structure + changepoint + anomalies)
feat["composite_risk_v2"] = ( 1.0 * zscore(feat["news_7d_z_pm"].fillna(0.0))
                              + 0.25 * feat["proba_ens"]
                              + 0.15 * zscore(feat["cp_score"].fillna(0.0))
                              - 0.4  * feat["LPI_z"].fillna(0.0)
                              - 0.4  * feat["WGI_MEAN_z"].fillna(0.0)
                              + 0.15 * zscore(feat["anom_ens"].fillna(0.0)) )

# outputs
risk = feat.sort_values("composite_risk_v2", ascending=False).reset_index(drop=True)
anom = feat.sort_values("anom_ens", ascending=False).reset_index(drop=True)
risk.to_csv(GOLD/"country_risk_composite_live.csv", index=False)
anom[["iso3","anom_ens","if_anom","ocsvm_anom","lof_anom"]].to_csv(GOLD/"country_risk_anomaly_live.csv", index=False)

# Profit-oriented alert threshold on proba_ens (proxy economics)
Cost_false_alert, Cost_missed_event, Benefit_caught = 10.0, 2000.0, 800.0
grid = np.linspace(0.05, 0.95, 19)
best_thr, best_profit = None, -1e9
for t in grid:
    yhat = (feat["proba_ens"].values >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    profit = tp*Benefit_caught - fp*Cost_false_alert - fn*Cost_missed_event
    if profit > best_profit:
        best_profit, best_thr = profit, float(t)

zT = 1.0
mu, sd = risk["composite_risk_v2"].mean(), (risk["composite_risk_v2"].std() or 1.0)
cut = mu + zT*sd
alerts = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "threshold_prob": best_thr,
    "profit_proxy": float(best_profit),
    "composite_field": "composite_risk_v2",
    "composite_cut_value": float(cut),
    "hits": risk[risk["composite_risk_v2"]>=cut].head(20).to_dict("records"),
    "anomaly_field": "anom_ens",
    "anomaly_top10": anom.head(10).to_dict("records"),
}
(GOLD/"alerts.json").write_text(json.dumps(alerts, indent=2))

# ----------------------------5) LkSG REGISTER (actionable)----------------------------

Z_HIGH, Z_MED = 1.0, 0.3
r2 = risk.copy()
mu2, sd2 = r2["composite_risk_v2"].mean(), (r2["composite_risk_v2"].std() or 1.0)
r2["z"] = (r2["composite_risk_v2"] - mu2) / sd2

def action_from_z(z):
    if z >= Z_HIGH: return "Enhanced Due Diligence"
    if z >= Z_MED:  return "Monitor & Mitigate"
    return "Baseline Due Diligence"

def rationale(row):
    bits=[]
    if row.get("news_7d_z_pm",0)>0.8: bits.append("7d incident spike")
    if row.get("news_7d_delta_pct",0)>0.5: bits.append("volume > baseline")
    if pd.notna(row.get("LPI",np.nan)) and row["LPI"]<3.2: bits.append("weak logistics")
    if pd.notna(row.get("WGI_MEAN",np.nan)) and row["WGI_MEAN"]<0: bits.append("governance")
    if pd.notna(row.get("cp_score",np.nan)) and row["cp_score"]>0.8: bits.append("recent change-point")
    return ", ".join(bits) if bits else "balanced"

PLAYBOOK = {
    "Enhanced Due Diligence": "Escalate questionnaire; review tier-2; require CAP; increase cadence.",
    "Monitor & Mitigate":     "Track weekly; request clarifications; add clauses; prep CAP if worsens.",
    "Baseline Due Diligence": "Standard checks; periodic monitoring."
}
r2["Action"] = r2["z"].apply(action_from_z)
r2["Rationale"] = r2.apply(rationale, axis=1)
r2["Suggested_Next_Step"] = r2["Action"].map(PLAYBOOK)
r2.to_csv(GOLD/"lkSG_risk_register.csv", index=False)

# ----------------------------6) NLP THEMES (parallel GDELT ArtList by country)----------------------------

QUERY_BASE = ("(port OR terminal OR vessel OR container OR customs OR rail OR truck OR highway OR factory "
              "OR strike OR protest OR union OR blockade OR sanctions OR tariff OR outage OR shutdown "
              "OR inspection OR recall OR wildfire OR flood OR storm OR cyclone OR typhoon OR earthquake "
              "OR ransomware OR cyber OR congestion OR backlog)")

def gdelt_doc_for_iso2(iso2: str, max_records=250) -> pd.DataFrame:
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {"query": f"{QUERY_BASE} AND sourcecountry:{iso2}",
              "mode":"ArtList","timespan":"7d","sort":"DateDesc","format":"JSON"}
    try:
        r = SESSION.get(url, params=params, timeout=45)
        r.raise_for_status()
        arts = r.json().get("articles", [])[:max_records]
        df = pd.DataFrame(arts)
        df["iso2"] = iso2
        return df
    except Exception:
        return pd.DataFrame()

def fetch_all_themes(iso3_to_iso2: dict, max_workers=8) -> pd.DataFrame:
    out=[]
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(gdelt_doc_for_iso2, iso2): iso3 for iso3, iso2 in iso3_to_iso2.items() if iso2}
        for fut in as_completed(futs):
            df = fut.result()
            if not df.empty:
                df["iso3"] = ISO2_TO_ISO3.get(df["iso2"].iloc[0])
                out.append(df)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

gdocs = fetch_all_themes({c: ISO3_TO_ISO2.get(c) for c in ISO3})

if gdocs.empty:
    tags_by_country = pd.DataFrame({"iso3": ISO3, "theme_top": "None", "theme_counts": ["{}"]*len(ISO3)})
else:
    gdocs.columns = [c.lower() for c in gdocs.columns]
    THEMES = {
        "Logistics":  r"\b(port|terminal|vessel|container|customs|clearance|rail|highway|truck|congestion|backlog|berth|anchorage)\b",
        "Labor":      r"\b(strike|walkout|union|picket|labor|labour|protest|demonstration|wage|collective bargaining)\b",
        "Environment":r"\b(flood|storm|cyclone|typhoon|wildfire|landslide|earthquake|heatwave|drought)\b",
        "Cyber":      r"\b(cyber|ransomware|malware|ddos|data breach|hack|phishing|encryption)\b",
        "Political":  r"\b(sanction|tariff|export control|embargo|visa|blockade|geopolit|election|regulation)\b",
        "Industrial": r"\b(outage|shutdown|fire|explosion|maintenance|inspection|recall|quality issue)\b"
    }
    def tag_title(title: str):
        t = str(title).lower()
        hits = [k for k,pat in THEMES.items() if re.search(pat, t)]
        return hits or ["Other"]
    gdocs["title"] = gdocs.get("title","").astype(str)
    gdocs["themes"] = gdocs["title"].apply(tag_title)
    exp = gdocs.explode("themes")
    cnt = (exp.groupby(["iso3","themes"]).size()
            .rename("n").reset_index()
            .sort_values(["iso3","n"], ascending=[True, False]))
    def top_theme(df):
        row = df.sort_values("n", ascending=False).head(1)
        return row["themes"].values[0] if len(row) else "Other"
    theme_top = cnt.groupby("iso3").apply(top_theme).reset_index(name="theme_top")
    theme_dicts = cnt.groupby("iso3").apply(lambda d: json.dumps(dict(zip(d["themes"], d["n"]))))
    theme_df = pd.DataFrame({"iso3": theme_dicts.index, "theme_counts": theme_dicts.values})
    tags_by_country = theme_top.merge(theme_df, on="iso3", how="left")

# merge into risk + register + BI file
risk_path = GOLD/"country_risk_composite_live.csv"
reg_path  = GOLD/"lkSG_risk_register.csv"
risk = pd.read_csv(risk_path)
risk = risk.merge(tags_by_country, on="iso3", how="left")
risk["theme_top"] = risk["theme_top"].fillna("None")
risk["theme_counts"] = risk["theme_counts"].fillna("{}")
risk.to_csv(risk_path, index=False)

if reg_path.exists():
    reg = pd.read_csv(reg_path).merge(tags_by_country, on="iso3", how="left")
    reg["theme_top"] = reg["theme_top"].fillna("None")
    reg["theme_counts"] = reg["theme_counts"].fillna("{}")
    reg.to_csv(reg_path, index=False)

bi_cols = ["iso3",
           "composite_risk_v2","LPI","WGI_MEAN",
           "news_7d_pm","news_7d_z_pm","news_7d_delta_pct",
           "cp_score","proba_ens","anom_ens","theme_top","theme_counts"]
bi_cols = [c for c in bi_cols if c in risk.columns]
risk[bi_cols].sort_values("composite_risk_v2", ascending=False)\
             .to_csv(GOLD/"bi_ready_country_risk_with_themes.csv", index=False)

# ----------------------------7) Plotly visuals (world map + anomalies bar)----------------------------

risk_vis = pd.read_csv(GOLD/"country_risk_composite_live.csv")
anom_vis = pd.read_csv(GOLD/"country_risk_anomaly_live.csv")
risk_vis["iso3"] = risk_vis["iso3"].astype(str)

fig_map = px.choropleth(
    risk_vis, locations="iso3", color="composite_risk_v2",
    color_continuous_scale="Reds", locationmode="ISO-3",
    hover_data={"iso3":True,"composite_risk_v2":":.2f","LPI":":.2f","WGI_MEAN":":.2f","theme_top":True},
    title="Composite Supply-Chain Risk v2 (live, higher = riskier)"
)
fig_map.update_layout(height=520)
fig_map.show()

topn = anom_vis[["iso3","anom_ens"]].head(12).copy()
topn["anom_ens"] = topn["anom_ens"].astype(float)
fig_bar = px.bar(topn, x="iso3", y="anom_ens", title="Top 12 anomaly scores (ensemble)")
fig_bar.update_layout(height=360)
fig_bar.show()

print("Pipeline run completed.")
print("   - Risk table:", (GOLD/'country_risk_composite_live.csv').as_posix())
print("   - Anomalies :", (GOLD/'country_risk_anomaly_live.csv').as_posix())
print("   - Alerts    :", (GOLD/'alerts.json').as_posix())
print("   - LkSG Reg  :", (GOLD/'lkSG_risk_register.csv').as_posix())
print("   - BI CSV    :", (GOLD/'bi_ready_country_risk_with_themes.csv').as_posix())

# ============================================================

import json, numpy as np, pandas as pd, pathlib, plotly.express as px
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from sklearn.inspection import permutation_importance

GOLD   = pathlib.Path("/content/live_project/data/gold")
risk   = pd.read_csv(GOLD/"country_risk_composite_live.csv")

# -------------------- data --------------------
# Features & weak labels used in the training block
X_cols = [c for c in ["news_7d_pm","news_7d_z_pm","news_7d_delta_pct","cp_score","LPI","WGI_MEAN"]
          if c in risk.columns]
assert len(X_cols)>0, "No feature columns found."
X = risk[X_cols].copy().fillna(risk[X_cols].median(numeric_only=True))
y = (risk.get("weak_label", (risk["composite_risk_v2"] >= risk["composite_risk_v2"].quantile(0.85)).astype(int))
     if "weak_label" in risk.columns else
     (risk["composite_risk_v2"] >= risk["composite_risk_v2"].quantile(0.85)).astype(int)).values

# Trained scores from the build step (if present)
have_rf   = "rf_proba"    in risk.columns
have_lgbm = "lgbm_proba"  in risk.columns
have_ens  = "proba_ens"   in risk.columns

# -------------------- CV evaluation (proxy) --------------------
# Re-train light models inside CV to get unbiased estimates
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def eval_cv(model_maker, X, y, name, splits=5, repeats=4, seed=42):
    rskf = RepeatedStratifiedKFold(n_splits=min(splits, sum(y)>=2 and sum(1-y)>=2 and 5 or 3),
                                   n_repeats=repeats, random_state=seed)
    prs, rocs, briers, logs = [], [], [], []
    for tr, te in rskf.split(X, y):
        m = model_maker()
        m.fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[te])[:,1]
        # Guard against degenerate splits
        if len(np.unique(y[te]))<2:
            continue
        prs.append(average_precision_score(y[te], p))
        rocs.append(roc_auc_score(y[te], p))
        briers.append(brier_score_loss(y[te], p))
        try:
            logs.append(log_loss(y[te], p, eps=1e-7))
        except Exception:
            pass
    return {
        "model": name,
        "PR_AUC_mean": float(np.mean(prs)) if prs else None,
        "ROC_AUC_mean": float(np.mean(rocs)) if rocs else None,
        "Brier_mean":   float(np.mean(briers)) if briers else None,
        "LogLoss_mean": float(np.mean(logs)) if logs else None,
        "folds": len(prs)
    }

def mk_rf():
    return RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample", random_state=42)

def mk_lgbm():
    n = len(X)
    return lgb.LGBMClassifier(
        objective="binary", n_estimators=400, learning_rate=0.05,
        num_leaves=min(15, max(2, n-1)), min_data_in_leaf=1,
        min_sum_hessian_in_leaf=1e-3, feature_fraction=1.0,
        bagging_fraction=1.0, reg_lambda=0.1, random_state=42
    )

def mk_meta():
    # meta on top of RF+LGBM: fit base → stack → logistic
    rf, lgbm = mk_rf(), mk_lgbm()
    class Stack:
        def fit(self, X, y):
            rf.fit(X, y); lgbm.fit(X, y)
            P = np.c_[rf.predict_proba(X)[:,1], lgbm.predict_proba(X)[:,1]]
            self.meta = LogisticRegression(max_iter=1000).fit(P, y)
            self.rf, self.lgbm = rf, lgbm
            return self
        def predict_proba(self, X):
            P = np.c_[self.rf.predict_proba(X)[:,1], self.lgbm.predict_proba(X)[:,1]]
            proba = self.meta.predict_proba(P)[:,1]
            return np.c_[1-proba, proba]
    return Stack()

report = []
report.append(eval_cv(mk_rf,   X, y, "RandomForest"))
report.append(eval_cv(mk_lgbm, X, y, "LightGBM"))
report.append(eval_cv(mk_meta, X, y, "RF+LGBM meta"))

# -------------------- permutation importances --------------------
# Fit once on full data for importances (quick diagnostic)
rf_full = mk_rf(); rf_full.fit(X, y)
lgb_full = mk_lgbm(); lgb_full.fit(X, y)

imp_rf  = permutation_importance(rf_full, X, y, n_repeats=50, random_state=42)
imp_lgb = permutation_importance(lgb_full, X, y, n_repeats=50, random_state=42)

imp_df = pd.DataFrame({
    "feature": X.columns,
    "RF_importance":  imp_rf.importances_mean,
    "LGBM_importance":imp_lgb.importances_mean,
})
imp_df["avg_rank"] = (imp_df["RF_importance"].rank(ascending=False)
                      + imp_df["LGBM_importance"].rank(ascending=False)) / 2.0
imp_df = imp_df.sort_values("avg_rank")

# -------------------- anomaly diagnostics (unsupervised) --------------------
anom_cols = [c for c in ["if_anom","ocsvm_anom","lof_anom","anom_ens"] if c in risk.columns]
anom_diag = {}
if anom_cols:
    A = risk[anom_cols].copy()
    # rank-correlation between detectors (higher = consistent anomalies)
    anom_diag["kendall_tau_matrix"] = A.rank().corr(method="kendall").round(3).to_dict()
    # correlation between anom_ens and composite_risk_v2 (do anomalies align with risk?)
    if "composite_risk_v2" in risk.columns and "anom_ens" in A.columns:
        anom_diag["anom_vs_risk_corr"] = float(risk["composite_risk_v2"].corr(A["anom_ens"]))

# -------------------- save & display --------------------
metrics = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "n_samples": int(len(X)),
    "features": X_cols,
    "cv_report": report,
    "permutation_importance": imp_df.to_dict("records"),
    "anomaly_diagnostics": anom_diag
}
(GOLD/"model_report.json").write_text(json.dumps(metrics, indent=2))

print("Metrics written to:", (GOLD/"model_report.json").as_posix())
display(pd.DataFrame(report))

# Plot importances
fig_imp = px.bar(imp_df.melt(id_vars="feature", value_vars=["RF_importance","LGBM_importance"],
                             var_name="model", value_name="importance"),
                 x="feature", y="importance", color="model",
                 title="Permutation importance (higher = more predictive)")
fig_imp.update_layout(height=420)
fig_imp.show()

# Quick calibration scatter using in-sample proba_ens if present
if have_ens:
    df_cal = pd.DataFrame({"y": y, "proba_ens": risk["proba_ens"].values})
    df_cal["bin"] = pd.qcut(df_cal["proba_ens"], q=min(10, max(3, len(df_cal)//3)), duplicates="drop")
    cal = df_cal.groupby("bin").agg(mean_pred=("proba_ens","mean"), frac_pos=("y","mean")).reset_index(drop=True)
    fig_cal = px.scatter(cal, x="mean_pred", y="frac_pos",
                         title="Calibration (proba_ens, weak-label proxy)")
    fig_cal.add_shape(type="line", x0=0, y0=0, x1=1, y1=1)
    fig_cal.update_layout(height=360)
    fig_cal.show()

# Show top rows with scores for quick “inference”
cols_show = ["iso3","composite_risk_v2","proba_ens","rf_proba","lgbm_proba","anom_ens"] + X_cols
cols_show = [c for c in cols_show if c in risk.columns]
display(risk[cols_show].head(12))



# Environment setup: install key libraries (keep NumPy and pandas from Colab)
!pip -q install --no-deps scikit-learn==1.5.2 lightgbm==4.5.0 xgboost==2.1.1 catboost==1.2.7 \
                       ruptures==1.1.9 hdbscan==0.8.33 plotly==5.24.1
!pip install catboost
import os, json, math, warnings, pathlib, numpy as np, pandas as pd, requests
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance

import lightgbm as lgb
try:
    import xgboost as xgb
except Exception:
    xgb = None
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None
import ruptures as rpt
try:
    import hdbscan
except Exception:
    hdbscan = None
import plotly.express as px

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ----------------------- paths (duplicate block - consider removing) -----------------------
BASE   = pathlib.Path("/content/live_project")
BRONZE = BASE/"data"/"bronze"; BRONZE.mkdir(parents=True, exist_ok=True)
SILVER = BASE/"data"/"silver"; SILVER.mkdir(parents=True, exist_ok=True)
GOLD   = BASE/"data"/"gold"  ; GOLD.mkdir(parents=True, exist_ok=True)
ART    = BASE/"artifacts"    ; ART.mkdir(parents=True, exist_ok=True)

# ----------------------- watchlist -------------------
ISO3 = ["DEU","NLD","POL","FRA","ITA","ESP","CZE","AUT","SWE","DNK","NOR","FIN",
        "GBR","IRL","USA","CAN","MEX","BRA","CHN","JPN","KOR","IND","THA","VNM","TUR"]
ISO2_TO_ISO3 = {
    "DE":"DEU","NL":"NLD","PL":"POL","FR":"FRA","IT":"ITA","ES":"ESP","CZ":"CZE","AT":"AUT","SE":"SWE",
    "DK":"DNK","NO":"NOR","FI":"FIN","GB":"GBR","UK":"GBR","IE":"IRL","US":"USA","CA":"CAN","MX":"MEX",
    "BR":"BRA","CN":"CHN","JP":"JPN","KR":"KOR","IN":"IND","TH":"THA","VN":"VNM","TR":"TUR"
}

print("UTC:", datetime.utcnow().strftime("%Y-%m-%d %H:%M"))

# ----------------------- World Bank (live, no key) -----------------------
WB_BASE = "https://api.worldbank.org/v2/country/{ISO}/indicator/{IND}?format=json"
def wb_fetch_indicator(iso_list, indicator, mrv=1):
    rows=[]
    for iso in iso_list:
        try:
            r = requests.get(WB_BASE.format(ISO=iso, IND=indicator),
                             params={"mrv":str(mrv)}, timeout=45)
            r.raise_for_status()
            js=r.json()
            if isinstance(js, list) and len(js)==2 and js[1]:
                for rec in js[1]:
                    if rec and rec.get("value") is not None:
                        rows.append({
                            "iso3": rec.get("countryiso3code"),
                            "date": int(rec.get("date")) if rec.get("date") else None,
                            "indicator": indicator,
                            "value": float(rec.get("value"))
                        })
        except Exception:
            pass
    return pd.DataFrame(rows)

LPI = "LP.LPI.OVRL.XQ"   # Logistics Performance Index
POP = "SP.POP.TOTL"      # Population
WGI_CODES = ["GE.EST","RQ.EST","RL.EST","CC.EST"]  # governance subindices (estimates)

print("Pulling World Bank…")
wb_lpi = wb_fetch_indicator(ISO3, LPI, mrv=1)
wb_pop = wb_fetch_indicator(ISO3, POP, mrv=1)
wb_wgi = pd.concat([wb_fetch_indicator(ISO3, c, mrv=1) for c in WGI_CODES], ignore_index=True)

for name, df in [("wb_lpi", wb_lpi), ("wb_pop", wb_pop), ("wb_wgi", wb_wgi)]:
    if not df.empty: df.to_parquet(BRONZE/f"{name}.parquet", index=False)

def latest_value(df, indicator, alias):
    if df.empty: return pd.DataFrame({"iso3": ISO3, alias: np.nan})
    d = df[df["indicator"]==indicator].copy()
    if d.empty: return pd.DataFrame({"iso3": ISO3, alias: np.nan})
    d = d.sort_values(["iso3","date"]).groupby("iso3").tail(1)
    out = d[["iso3","value"]].rename(columns={"value": alias})
    missing = [c for c in ISO3 if c not in set(out["iso3"])]
    if missing: out = pd.concat([out, pd.DataFrame({"iso3": missing, alias: np.nan})], ignore_index=True)
    return out

lpi_latest = latest_value(wb_lpi, LPI, "LPI")
pop_latest = latest_value(wb_pop, POP, "POP")

if not wb_wgi.empty:
    wgi = (wb_wgi.sort_values(["iso3","date"])
                 .groupby(["iso3","indicator"]).tail(1)
                 .pivot(index="iso3", columns="indicator", values="value")
                 .reset_index())
    for c in WGI_CODES:
        if c not in wgi.columns: wgi[c]=np.nan
    wgi["WGI_MEAN"] = wgi[WGI_CODES].mean(axis=1)
    wgi = wgi[["iso3","WGI_MEAN"]]
else:
    wgi = pd.DataFrame({"iso3": ISO3, "WGI_MEAN": np.nan})

# ----------------------- GDELT (live, no key) -----------------------
QUERY = ("(port OR strike OR flood OR storm OR customs OR wildfire OR landslide OR earthquake OR cyber) "
         "AND (delay OR disruption OR shutdown OR congestion OR outage OR blockade)")

def gdelt_timeline_matrix(query, timespan="30d"):
    try:
        r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc",
                         params={"query": query, "mode":"timelinesourcecountry",
                                 "timespan": timespan, "format":"JSON"},
                         timeout=45)
        r.raise_for_status()
        js = r.json()
        tl = js.get("timeline", [])
        return pd.DataFrame(tl) if tl else pd.DataFrame()
    except Exception:
        return pd.DataFrame()

def ensure_date_series(df: pd.DataFrame) -> pd.Series:
    for cand in ("date","datetime","time","timestamp"):
        if cand in df.columns:
            s = pd.to_datetime(df[cand].astype(str), errors="coerce")
            return s.dt.normalize()
    end = pd.Timestamp.utcnow().normalize()
    start = end - pd.Timedelta(days=max(len(df)-1, 0))
    return pd.Series(pd.date_range(start, end, freq="D"), index=df.index)

print("Pulling GDELT 30d timeline…")
raw = gdelt_timeline_matrix(QUERY, "30d")

# Build 30d matrix (raw + per-million)
if raw.empty:
    dates = pd.date_range(pd.Timestamp.utcnow().normalize()-pd.Timedelta(days=29),
                          pd.Timestamp.utcnow().normalize(), freq="D")
    mat = pd.DataFrame(0.0, index=dates, columns=ISO3)
else:
    raw["date"] = ensure_date_series(raw)
    iso2_cols = [c for c in raw.columns if c != "date"]
    long = raw.melt(id_vars=["date"], value_vars=iso2_cols, var_name="iso2", value_name="count").fillna(0)
    long["iso3"] = long["iso2"].str.upper().map(ISO2_TO_ISO3)
    long = long.dropna(subset=["iso3"])
    long = long[long["iso3"].isin(ISO3)]
    mat = long.pivot_table(index="date", columns="iso3", values="count", aggfunc="sum").sort_index()
    for c in ISO3:
        if c not in mat.columns: mat[c]=0.0
    end = mat.index.max() if len(mat.index) else pd.Timestamp.utcnow().normalize()
    start = end - pd.Timedelta(days=29)
    mat = mat.reindex(pd.date_range(start, end, freq="D"), fill_value=0.0)
    mat = mat[ISO3]

mat.astype(float).to_parquet(SILVER/"gdelt_counts_30d.parquet")

# per-million
POP_MAP = dict(zip(pop_latest["iso3"], pop_latest["POP"])) if not pop_latest.empty else {}
divisors = pd.Series({c: (POP_MAP.get(c, np.nan)/1_000_000.0) for c in mat.columns})
if divisors.notna().any():
    fill_val = float(divisors.dropna().median())
    divisors = divisors.fillna(fill_val if not np.isnan(fill_val) else 1.0)
else:
    divisors = pd.Series(1.0, index=mat.columns)
mat_pm = mat.divide(divisors, axis=1).astype(float)
mat_pm.to_parquet(SILVER/"gdelt_counts_pm_30d.parquet")

# 7d spike vs 30d baseline
pm_7d  = mat_pm.rolling(window=7,  min_periods=1).sum().iloc[-1]
pm_30m = mat_pm.rolling(window=30, min_periods=7).mean().iloc[-1]
pm_30s = mat_pm.rolling(window=30, min_periods=7).std(ddof=0).iloc[-1]
eps = 1e-6
news_7d_pm        = pm_7d
news_7d_z_pm      = (pm_7d - (pm_30m*7.0)) / (np.sqrt((pm_30s**2)*7.0) + eps)
news_7d_delta_pct = (pm_7d - (pm_30m*7.0)) / ((pm_30m*7.0) + eps)

news_feats = pd.DataFrame({
    "iso3": mat_pm.columns,
    "news_7d_pm": news_7d_pm.values.astype(float),
    "news_7d_z_pm": news_7d_z_pm.values.astype(float),
    "news_7d_delta_pct": news_7d_delta_pct.values.astype(float),
}).sort_values("iso3").reset_index(drop=True)
news_feats.to_csv(SILVER/"news_feats_gdelt_pm.csv", index=False)

# ----------------------- change-points -----------------------
def changepoint_score(series: pd.Series) -> float:
    y = series.values.astype(float)
    if np.allclose(y, y[0]) or len(y)<15:
        return 0.0
    try:
        algo = rpt.Pelt(model="rbf").fit(y)
        bkps = algo.predict(pen=1.0)
        if len(bkps) <= 1:
            return 0.0
        last_cp = sorted(bkps[:-1])[-1]
        pre = y[max(0,last_cp-7):last_cp]
        post = y[last_cp:min(len(y), last_cp+7)]
        if len(pre)==0 or len(post)==0:
            return 0.0
        diff = np.mean(post) - np.mean(pre)
        std  = np.std(y) if np.std(y)>0 else 1.0
        return float(diff / std)
    except Exception:
        return 0.0

cp_scores = {c: changepoint_score(mat_pm[c]) for c in mat_pm.columns}
cp_df = pd.DataFrame({"iso3": list(cp_scores.keys()), "cp_score": list(cp_scores.values())})
cp_df.to_csv(SILVER/"gdelt_changepoint_scores.csv", index=False)

# ----------------------- feature table -----------------------
def z(col):
    sd = col.std()
    return (col - col.mean()) / (sd if (sd and sd>0) else 1.0)

feat = (lpi_latest.merge(wgi, on="iso3", how="left")
                 .merge(pop_latest, on="iso3", how="left")
                 .merge(news_feats, on="iso3", how="left")
                 .merge(cp_df, on="iso3", how="left"))

feat["LPI_z"] = z(feat["LPI"])
feat["WGI_MEAN_z"] = z(feat["WGI_MEAN"])
feat["composite_risk_v1"] = ( 1.3*feat["news_7d_z_pm"].fillna(0.0)
                            + 0.3*feat["news_7d_delta_pct"].fillna(0.0)
                            - 0.6*feat["LPI_z"].fillna(0.0)
                            - 0.6*feat["WGI_MEAN_z"].fillna(0.0))

# weak labels for ranking
weak_q = 0.85
thr = float(np.quantile(feat["composite_risk_v1"], weak_q))
feat["weak_label"] = (feat["composite_risk_v1"] >= thr).astype(int)

X_cols = ["news_7d_pm","news_7d_z_pm","news_7d_delta_pct","cp_score","LPI","WGI_MEAN"]
X = feat[X_cols].copy().fillna(feat[X_cols].median(numeric_only=True))
y = feat["weak_label"].values
n = len(X)

# ----------------------- classifiers (in-sample fit) -----------------------
# RF
rf = RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample", random_state=42)
rf.fit(X, y)
feat["rf_proba"] = rf.predict_proba(X)[:,1]

# LightGBM
lgbm = lgb.LGBMClassifier(
    objective="binary", n_estimators=600, learning_rate=0.05,
    num_leaves=min(31, max(2, n-1)),
    min_data_in_leaf=1, min_sum_hessian_in_leaf=1e-3,
    feature_fraction=1.0, bagging_fraction=1.0, reg_lambda=0.1,
    random_state=42
)
lgbm.fit(X, y)
feat["lgbm_proba"] = lgbm.predict_proba(X)[:,1]

# XGBoost (CPU) — optional if libgomp/OpenMP missing on Colab
if xgb is not None:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=600, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        objective="binary:logistic", tree_method="hist", predictor="cpu_predictor",
        random_state=42
    )
    xgb_clf.fit(X, y)
    feat["xgb_proba"] = xgb_clf.predict_proba(X)[:,1]
else:
    feat["xgb_proba"] = np.nan  # placeholder so downstream columns exist

# CatBoost (CPU, silent) — optional if numpy ABI mismatch on Colab
if CatBoostClassifier is not None:
    cat = CatBoostClassifier(
        iterations=800, depth=4, learning_rate=0.05, loss_function="Logloss",
        random_seed=42, verbose=False, allow_writing_files=False, task_type="CPU"
    )
    cat.fit(X, y)
    feat["cat_proba"] = cat.predict_proba(X)[:,1]
else:
    feat["cat_proba"] = np.nan  # placeholder so downstream columns exist

# Stacked (simple average, in-sample)
clf_cols = ["rf_proba", "lgbm_proba"] + (["xgb_proba"] if xgb is not None else []) + (["cat_proba"] if CatBoostClassifier is not None else [])
feat["proba_ens"] = feat[[c for c in clf_cols if c in feat.columns]].mean(axis=1)

# ----------------------- anomalies -----------------------
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

iso = IsolationForest(n_estimators=400, contamination=0.15, random_state=42).fit(Xs)
feat["if_anom"] = -iso.score_samples(Xs)

ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.15).fit(Xs)
feat["ocsvm_anom"] = -ocsvm.score_samples(Xs)

lof = LocalOutlierFactor(n_neighbors=min(10, max(5, n-1)), contamination=0.15)
_ = lof.fit_predict(Xs)
feat["lof_anom"] = -lof.negative_outlier_factor_

# HDBSCAN outlier scores (robust to version differences); skip if import failed (e.g. numpy ABI on Colab)
try:
    if hdbscan is None:
        feat["hdb_outlier"] = 0.0
    else:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=max(3, n // 5),
            min_samples=1,
            prediction_data=True
        )
        _labels = clusterer.fit_predict(Xs)
        outlier_scores = None
        if hasattr(hdbscan, "outlier_scores"):
            outlier_scores = hdbscan.outlier_scores(clusterer)
        if outlier_scores is None or np.asarray(outlier_scores).size != len(Xs):
            soft = hdbscan.all_points_membership_vectors(clusterer)
            if soft is not None and soft.shape[0] == len(Xs):
                outlier_scores = 1.0 - soft.max(axis=1)
            else:
                outlier_scores = np.zeros(len(Xs), dtype=float)
        feat["hdb_outlier"] = pd.Series(outlier_scores, index=feat.index).fillna(0.0)
except Exception:
    feat["hdb_outlier"] = 0.0

# anomaly ensemble (includes HDBSCAN channel)
anom_parts = [feat["if_anom"], feat["ocsvm_anom"], feat["lof_anom"], feat["hdb_outlier"]]
feat["anom_ens"] = np.mean(np.vstack([p.values for p in anom_parts]), axis=0)

# new composite using ensemble proba + changepoints + structure
feat["composite_risk_v2"] = ( 1.0 * z(feat["news_7d_z_pm"].fillna(0.0))
                              + 0.25 * feat["proba_ens"]
                              + 0.15 * z(feat["cp_score"].fillna(0.0))
                              - 0.4  * feat["LPI_z"].fillna(0.0)
                              - 0.4  * feat["WGI_MEAN_z"].fillna(0.0)
                              + 0.15 * z(feat["anom_ens"].fillna(0.0)) )

# ----------------------- metrics on weak labels (in-sample) -----------------------
def safe_metric(fn, y_true, y_score):
    try:
        return float(fn(y_true, y_score))
    except Exception:
        return float("nan")

metrics = {}
_metric_names = ["rf_proba", "lgbm_proba"] + (["xgb_proba"] if xgb is not None else []) + (["cat_proba"] if CatBoostClassifier is not None else []) + ["proba_ens"]
for name in _metric_names:
    if name not in feat.columns:
        continue
    pr  = safe_metric(average_precision_score, y, feat[name].values)
    roc = safe_metric(roc_auc_score, y, feat[name].values)
    b   = safe_metric(brier_score_loss, y, feat[name].values)
    metrics[name] = {"pr_auc": pr, "roc_auc": roc, "brier": b}

# profit-optimal alert threshold on in-sample ensemble proba (proxy economics)
Cost_false_alert   = 10.0
Cost_missed_event  = 2000.0
Benefit_caught     = 800.0
proba = feat["proba_ens"].values
grid = np.linspace(0.05, 0.95, 19)
best_thr, best_profit = None, -1e9
for t in grid:
    yhat = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    profit = tp*Benefit_caught - fp*Cost_false_alert - fn*Cost_missed_event
    if profit > best_profit:
        best_profit, best_thr = profit, float(t)

# ----------------------- OOF (out-of-fold) evaluation + importance -----------------------
RNG = 42
cv = StratifiedKFold(n_splits=min(5, max(3, sum(y))), shuffle=True, random_state=RNG)

# Slightly regularized clones for OOF to reduce tiny-N overfit
rf_oof  = RandomForestClassifier(n_estimators=400, max_depth=4, min_samples_leaf=2,
                                 class_weight="balanced_subsample", random_state=RNG)
lgbm_oof = lgb.LGBMClassifier(objective="binary", n_estimators=400, learning_rate=0.05,
                              num_leaves=min(15, max(2, n-1)), min_data_in_leaf=2,
                              reg_lambda=0.5, feature_fraction=0.9, bagging_fraction=0.9,
                              random_state=RNG)
if xgb is not None:
    xgb_oof = xgb.XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                                objective="binary:logistic", tree_method="hist",
                                predictor="cpu_predictor", random_state=RNG)
if CatBoostClassifier is not None:
    cat_oof = CatBoostClassifier(iterations=600, depth=3, learning_rate=0.05,
                                 l2_leaf_reg=6.0, loss_function="Logloss",
                                 random_seed=RNG, verbose=False, allow_writing_files=False)

models_cv = {"rf": rf_oof, "lgbm": lgbm_oof}
if xgb is not None:
    models_cv["xgb"] = xgb_oof
if CatBoostClassifier is not None:
    models_cv["cat"] = cat_oof
oof = {k: np.zeros(len(X), dtype=float) for k in models_cv}

for tr, te in cv.split(X, y):
    Xtr, Xte, ytr = X.iloc[tr], X.iloc[te], y[tr]
    for name, mdl in models_cv.items():
        m = mdl.__class__(**mdl.get_params())
        m.fit(Xtr, ytr)
        oof[name][te] = m.predict_proba(Xte)[:,1]

# OOF ensemble = mean of model OOF probs (use only available models)
_oof_keys = list(models_cv.keys())
oof["ens"] = np.mean(np.vstack([oof[k] for k in _oof_keys]), axis=0)
feat["oof_ens_proba"] = oof["ens"]

# OOF metrics (realistic)
metrics_oof = {}
for name, pred in oof.items():
    metrics_oof[name] = {
        "pr_auc":  safe_metric(average_precision_score, y, pred),
        "roc_auc": safe_metric(roc_auc_score, y, pred),
        "brier":   safe_metric(brier_score_loss, y, pred)
    }

# OOF profit threshold
best_thr_oof, best_profit_oof = None, -1e9
for t in grid:
    yhat = (oof["ens"] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    profit = tp*Benefit_caught - fp*Cost_false_alert - fn*Cost_missed_event
    if profit > best_profit_oof:
        best_profit_oof, best_thr_oof = profit, float(t)

# Permutation importance on LGBM_OOF retrained on all data
lgbm_oof.fit(X, y)
perm = permutation_importance(lgbm_oof, X, y, n_repeats=200, random_state=RNG, scoring="average_precision")
imp = (pd.DataFrame({"feature": X.columns, "mean_importance": perm.importances_mean,
                     "std": perm.importances_std})
       .sort_values("mean_importance", ascending=False))
imp.to_csv(ART/"permutation_importance_lgbm.csv", index=False)

# ----------------------- outputs -----------------------
risk = feat.sort_values("composite_risk_v2", ascending=False).reset_index(drop=True)
anom = feat.sort_values("anom_ens", ascending=False).reset_index(drop=True)

risk.to_csv(GOLD/"country_risk_composite_live.csv", index=False)
anom[["iso3","anom_ens","if_anom","ocsvm_anom","lof_anom","hdb_outlier"]].to_csv(GOLD/"country_risk_anomaly_live.csv", index=False)

zT = 1.0
mu, sd = risk["composite_risk_v2"].mean(), (risk["composite_risk_v2"].std() or 1.0)
cut = mu + zT*sd
alerts = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "threshold_prob_in_sample": best_thr,
    "profit_proxy_in_sample": float(best_profit),
    "threshold_prob_oof": best_thr_oof,
    "profit_proxy_oof": float(best_profit_oof),
    "composite_field": "composite_risk_v2",
    "composite_cut_value": float(cut),
    "hits": risk[risk["composite_risk_v2"]>=cut].head(20).to_dict("records"),
    "anomaly_field": "anom_ens",
    "anomaly_top10": anom.head(10).to_dict("records"),
}
(GOLD/"alerts.json").write_text(json.dumps(alerts, indent=2))

report_in = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "models_in_sample": metrics,
    "features_used": X_cols,
    "notes": {
        "weak_labels": f"top {int((1-weak_q)*100)}% by composite_v1 used as positive class",
        "composite_v2": "mix of z(news spike), ensemble proba, change-points, governance/logistics, anomaly ensemble",
        "economics": {"Cost_false_alert": Cost_false_alert, "Cost_missed_event": Cost_missed_event, "Benefit_caught": Benefit_caught}
    }
}
(ART/"model_report.json").write_text(json.dumps(report_in, indent=2))

report_oof = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "oof_metrics": metrics_oof,
    "oof_profit_threshold": {"threshold": best_thr_oof, "proxy_profit": best_profit_oof},
    "features_used": X_cols
}
(ART/"model_report_oof.json").write_text(json.dumps(report_oof, indent=2))

print("Files written successfully:")
print(" -", (GOLD/"country_risk_composite_live.csv").as_posix())
print(" -", (GOLD/"country_risk_anomaly_live.csv").as_posix())
print(" -", (GOLD/"alerts.json").as_posix())
print(" -", (ART/"model_report.json").as_posix())
print(" -", (ART/"model_report_oof.json").as_posix())
print(" -", (ART/"permutation_importance_lgbm.csv").as_posix())

# ----------------------- LkSG register -----------------------
Z_HIGH = 1.0
Z_MED  = 0.3
r2 = risk.copy()
mu2, sd2 = r2["composite_risk_v2"].mean(), (r2["composite_risk_v2"].std() or 1.0)
r2["z"] = (r2["composite_risk_v2"] - mu2) / sd2
def action_from_z(z):
    if z >= Z_HIGH: return "Enhanced Due Diligence"
    if z >= Z_MED:  return "Monitor & Mitigate"
    return "Baseline Due Diligence"
def rationale(row):
    bits=[]
    if row.get("news_7d_z_pm",0)>0.8: bits.append("7d incident spike")
    if row.get("news_7d_delta_pct",0)>0.5: bits.append("volume > baseline")
    if pd.notna(row.get("LPI",np.nan)) and row["LPI"]<3.2: bits.append("weak logistics")
    if pd.notna(row.get("WGI_MEAN",np.nan)) and row["WGI_MEAN"]<0: bits.append("governance")
    if pd.notna(row.get("cp_score",np.nan)) and row["cp_score"]>0.8: bits.append("recent change-point")
    return ", ".join(bits) if bits else "balanced"
r2["Action"] = r2["z"].apply(action_from_z)
r2["Rationale"] = r2.apply(rationale, axis=1)
PLAYBOOK = {
    "Enhanced Due Diligence": "Escalate questionnaire; review tier-2; require CAP; increase cadence.",
    "Monitor & Mitigate":     "Track weekly; request clarifications; add clauses; prep CAP if worsens.",
    "Baseline Due Diligence": "Standard checks; periodic monitoring."
}
r2["Suggested_Next_Step"] = r2["Action"].map(PLAYBOOK)
r2.to_csv(GOLD/"lkSG_risk_register.csv", index=False)
print("LkSG register written at:", (GOLD/'lkSG_risk_register.csv').as_posix())

# ----------------------- Plotly viz -----------------------
risk["iso3"] = risk["iso3"].astype(str)
fig_map = px.choropleth(
    risk, locations="iso3", color="composite_risk_v2",
    color_continuous_scale="Reds", locationmode="ISO-3",
    hover_data={"iso3":True,"composite_risk_v2":":.2f","LPI":":.2f","WGI_MEAN":":.2f"},
    title="Composite Supply-Chain Risk v2 (live, higher = riskier)"
)
fig_map.update_layout(height=520)
fig_map.show()

topn = anom[["iso3","anom_ens"]].head(12).copy()
topn["anom_ens"] = topn["anom_ens"].astype(float)
fig_bar = px.bar(topn, x="iso3", y="anom_ens", title="Top 12 anomaly scores (ensemble)")
fig_bar.update_layout(height=360)
fig_bar.show()

# Quick peek at metrics
from pprint import pprint
print("\nWeak-label metrics (in-sample; higher PR/ROC, lower Brier is better):")
pprint(metrics)
print(f"\nIn-sample profit-optimal probability threshold: {best_thr:.2f} (proxy profit={best_profit:.1f})")

print("\nOut-of-fold metrics (more realistic estimate):")
pprint(metrics_oof)
print(f"\nOOF profit-optimal probability threshold: {best_thr_oof:.2f} (proxy profit={best_profit_oof:.1f})")


# ----------------------------BI and Insights add-on (run after the main pipeline)----------------------------

risk_path   = GOLD/"country_risk_composite_live.csv"
anom_path   = GOLD/"country_risk_anomaly_live.csv"
counts_pm_p = SILVER/"gdelt_counts_pm_30d.parquet"

risk = pd.read_csv(risk_path)
anom = pd.read_csv(anom_path)
risk["iso3"] = risk["iso3"].astype(str)

# Graceful presence checks
def col(c): return c in risk.columns
# Expected columns from your pipeline (use defaults if missing)
for c, default in [
    ("composite_risk_v2", np.nan), ("LPI", np.nan), ("WGI_MEAN", np.nan),
    ("news_7d_pm", 0.0), ("news_7d_z_pm", 0.0), ("news_7d_delta_pct", 0.0),
    ("cp_score", 0.0), ("proba_ens", 0.0), ("anom_ens", 0.0)
]:
    if c not in risk.columns:
        risk[c] = default

# ---- KPIs & rank tables ----
risk_sorted    = risk.sort_values("composite_risk_v2", ascending=False).reset_index(drop=True)
anom_sorted    = risk.sort_values("anom_ens", ascending=False).reset_index(drop=True)
cp_sorted      = risk.sort_values("cp_score", ascending=False).reset_index(drop=True)
topN = 10

kpis = {
    "generated_at": pd.Timestamp.utcnow().isoformat()+"Z",
    "portfolio": {
        "avg_composite_risk": float(np.nanmean(risk["composite_risk_v2"])),
        "avg_proba_ens": float(np.nanmean(risk["proba_ens"])),
        "avg_anomaly": float(np.nanmean(risk["anom_ens"])),
        "n_countries": int(risk.shape[0]),
    },
    "top_risk": risk_sorted[["iso3","composite_risk_v2","proba_ens","cp_score","anom_ens"]].head(topN).to_dict("records"),
    "top_anomaly": anom_sorted[["iso3","anom_ens","proba_ens","composite_risk_v2"]].head(topN).to_dict("records"),
    "top_movers_cp": cp_sorted[["iso3","cp_score","news_7d_z_pm","composite_risk_v2"]].head(topN).to_dict("records"),
}
(ART/"bi_kpis.json").write_text(json.dumps(kpis, indent=2))

# A compact BI table for downstream tools
bi_cols = [c for c in [
    "iso3","composite_risk_v2","proba_ens","anom_ens","cp_score",
    "news_7d_pm","news_7d_z_pm","news_7d_delta_pct","LPI","WGI_MEAN"
] if c in risk.columns]
risk[bi_cols].to_csv(ART/"bi_table.csv", index=False)

# ---- Visual 1: Top-N composite risk bar ----
fig_top = px.bar(
    risk_sorted.head(topN),
    x="iso3", y="composite_risk_v2",
    hover_data={"proba_ens":":.3f","anom_ens":":.3f","cp_score":":.2f"},
    title=f"Top {topN} Composite Risk"
)
fig_top.update_layout(height=420)
fig_top.write_html(ART/"bi_top_risk.html")

# ---- Visual 2: Risk × Anomaly quadrant ----
fig_quad = px.scatter(
    risk, x="composite_risk_v2", y="anom_ens", text="iso3",
    hover_data={"proba_ens":":.3f","cp_score":":.2f","news_7d_z_pm":":.2f"},
    title="Quadrant: Composite Risk vs Anomaly"
)
fig_quad.update_traces(mode="markers+text", textposition="top center")
# draw medians
xr, yr = np.nanmedian(risk["composite_risk_v2"]), np.nanmedian(risk["anom_ens"])
fig_quad.add_vline(x=xr, line_dash="dash", opacity=0.4)
fig_quad.add_hline(y=yr, line_dash="dash", opacity=0.4)
fig_quad.update_layout(height=520)
fig_quad.write_html(ART/"bi_quadrant_risk_anomaly.html")

# ---- Visual 3: Feature heatmap (standardized) ----
heat_cols = [c for c in ["news_7d_z_pm","news_7d_delta_pct","cp_score","LPI","WGI_MEAN","proba_ens","anom_ens"] if c in risk.columns]
heat = risk[["iso3"]+heat_cols].set_index("iso3").copy()
heat_z = (heat - heat.mean()) / (heat.std().replace(0,1))
fig_heat = px.imshow(
    heat_z, aspect="auto", color_continuous_scale="RdBu_r",
    title="Feature Heatmap (z-scored across countries)"
)
fig_heat.update_layout(height=620)
fig_heat.write_html(ART/"bi_feature_heatmap.html")

# ---- Visual 4: News intensity sparklines (last 30 days, per-million) ----
spark_ok = counts_pm_p.exists()
if spark_ok:
    mat_pm = pd.read_parquet(counts_pm_p)     # index: dates, columns: iso3
    # reshape to long for plotly
    s_long = mat_pm.reset_index(names="date").melt(id_vars="date", var_name="iso3", value_name="count_pm")
    # keep only countries we scored
    s_long = s_long[s_long["iso3"].isin(risk["iso3"])]
    # build small multiples
    fig_spark = px.line(
        s_long, x="date", y="count_pm", facet_col="iso3", facet_col_wrap=6,
        title="News Mentions per-Million (last 30 days)"
    )
    fig_spark.update_layout(height=900, showlegend=False)
    fig_spark.write_html(ART/"bi_news_sparklines.html")

# ---- Country-level inferences (drivers) ----
# Decompose composite_risk_v2 into contributions (aligned with your formula)
def explain_row(row):
    parts = {}
    parts["news_spike_z"]   = float((row.get("news_7d_z_pm",0) - risk["news_7d_z_pm"].mean()) /
                                    (risk["news_7d_z_pm"].std() or 1.0))
    parts["cp_z"]           = float((row.get("cp_score",0) - risk["cp_score"].mean()) /
                                    (risk["cp_score"].std() or 1.0))
    parts["anom_z"]         = float((row.get("anom_ens",0) - risk["anom_ens"].mean()) /
                                    (risk["anom_ens"].std() or 1.0))
    parts["LPI_z"]          = float((row.get("LPI",np.nan) - risk["LPI"].mean()) /
                                    (risk["LPI"].std() or 1.0)) if pd.notna(row.get("LPI",np.nan)) else 0.0
    parts["WGI_MEAN_z"]     = float((row.get("WGI_MEAN",np.nan) - risk["WGI_MEAN"].mean()) /
                                    (risk["WGI_MEAN"].std() or 1.0)) if pd.notna(row.get("WGI_MEAN",np.nan)) else 0.0
    # weights from your composite_v2
    contrib = {
        "Incident spike (z)":         1.0  * parts["news_spike_z"],
        "Change-point (z)":           0.15 * parts["cp_z"],
        "Anomaly ensemble (z)":       0.15 * parts["anom_z"],
        "Governance (-)":            -0.40 * parts["WGI_MEAN_z"],
        "Logistics (-)":             -0.40 * parts["LPI_z"],
        "Model ensemble (proba)":     0.25 * float(row.get("proba_ens",0))
    }
    # Top 3 drivers by absolute impact
    top3 = sorted(contrib.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    return [{"factor": k, "impact": float(v)} for k,v in top3]

insights = []
for _, r in risk_sorted.head(12).iterrows():
    insights.append({
        "iso3": r["iso3"],
        "composite_risk_v2": float(r["composite_risk_v2"]),
        "proba_ens": float(r.get("proba_ens", np.nan)),
        "anom_ens": float(r.get("anom_ens", np.nan)),
        "cp_score": float(r.get("cp_score", np.nan)),
        "top_drivers": explain_row(r)
    })
(ART/"bi_top12_drivers.json").write_text(json.dumps(insights, indent=2))

# ---- Print an executive summary in the notebook ----
def pct(x):
    try: return f"{100*x:.1f}%"
    except: return "—"

print("==== Executive Summary ====")
print(f"Portfolio size: {kpis['portfolio']['n_countries']}")
print(f"Avg composite risk: {kpis['portfolio']['avg_composite_risk']:.3f} | "
      f"Avg model proba: {kpis['portfolio']['avg_proba_ens']:.3f} | "
      f"Avg anomaly: {kpis['portfolio']['avg_anomaly']:.3f}\n")

print("Top risk countries:")
for r in kpis["top_risk"][:5]:
    print(f" - {r['iso3']}: risk={r['composite_risk_v2']:.2f}, "
          f"proba={r['proba_ens']:.2f}, anom={r['anom_ens']:.2f}, cp={r['cp_score']:.2f}")

print("\nFastest movers (change-point):")
for r in kpis["top_movers_cp"][:5]:
    print(f" - {r['iso3']}: cp={r['cp_score']:.2f}, news_z={r['news_7d_z_pm']:.2f}, risk={r['composite_risk_v2']:.2f}")

print("\nKey drivers for top 5:")
for d in insights[:5]:
    drivers = "; ".join([f"{x['factor']} ({x['impact']:+.2f})" for x in d["top_drivers"]])
    print(f" - {d['iso3']}: {drivers}")

print("\nSaved BI artifacts:")
print("  • KPIs:                 ", (ART/'bi_kpis.json').as_posix())
print("  • BI table (CSV):       ", (ART/'bi_table.csv').as_posix())
print("  • Top risk bar (HTML):  ", (ART/'bi_top_risk.html').as_posix())
print("  • Risk×Anomaly (HTML):  ", (ART/'bi_quadrant_risk_anomaly.html').as_posix())
print("  • Heatmap (HTML):       ", (ART/'bi_feature_heatmap.html').as_posix())
if spark_ok:
    print("  • Sparklines (HTML):    ", (ART/'bi_news_sparklines.html').as_posix())
print("  • Top-12 drivers JSON:  ", (ART/'bi_top12_drivers.json').as_posix())

# ----------------------------BU-level BI pack (enhanced and fixed)----------------------------
risk_p   = GOLD/"country_risk_composite_live.csv"
anom_p   = GOLD/"country_risk_anomaly_live.csv"
counts_p = SILVER/"gdelt_counts_pm_30d.parquet"
sup_p    = INT/"supplier_master.csv"

risk = pd.read_csv(risk_p) if risk_p.exists() else pd.DataFrame()
anom = pd.read_csv(anom_p) if anom_p.exists() else pd.DataFrame()

# Ensure all expected columns
for c, default in [
    ("iso3",""), ("composite_risk_v2", np.nan), ("proba_ens", np.nan), ("anom_ens", np.nan),
    ("cp_score", np.nan), ("LPI", np.nan), ("WGI_MEAN", np.nan), ("news_7d_z_pm", 0.0), ("news_7d_delta_pct", 0.0)
]:
    if c not in risk.columns: risk[c] = default
risk["iso3"] = risk["iso3"].astype(str)

# Create template supplier_master if missing
if not sup_p.exists():
    tmpl = pd.DataFrame({
        "supplier_id":[f"S{i:03d}" for i in range(1,11)],
        "supplier_name":[f"Supplier_{i:02d}" for i in range(1,11)],
        "iso3":["MEX","BRA","TUR","VNM","IND","DEU","FRA","USA","CHN","POL"],
        "business_unit":["Logistics","Manufacturing","Procurement","Manufacturing","Procurement",
                         "Logistics","Compliance","Procurement","Manufacturing","Logistics"],
        "spend":[2_000_000,1_200_000,850_000,650_000,400_000,350_000,300_000,250_000,200_000,150_000],
        "criticality":["High","High","Medium","High","Medium","Low","Medium","Low","High","Low"]
    })
    tmpl.to_csv(sup_p, index=False)
    print(f"Created template supplier_master at: {sup_p}")

sup = pd.read_csv(sup_p)
for c in ["supplier_id","supplier_name","iso3","business_unit","criticality"]:
    if c not in sup.columns: sup[c] = ""
if "spend" not in sup.columns: sup["spend"] = 0.0
sup["iso3"] = sup["iso3"].astype(str)

# ----------------------------1) Supplier-level merge----------------------------
merged = sup.merge(risk, on="iso3", how="left", suffixes=("",""))
for c in ["composite_risk_v2","proba_ens","anom_ens","cp_score","LPI","WGI_MEAN"]:
    if c not in merged.columns: merged[c] = np.nan

def risk_bucket(x):
    if pd.isna(x): return "Unknown"
    if x >= 1.0: return "Severe"
    if x >= 0.6: return "High"
    if x >= 0.3: return "Moderate"
    if x >= 0.1: return "Low"
    return "Very Low"
merged["risk_bucket"] = merged["composite_risk_v2"].apply(risk_bucket)
merged["spend_at_risk"] = merged["spend"] * merged["proba_ens"].fillna(0)
merged.to_csv(BUART/"suppliers_with_risk.csv", index=False)

# ----------------------------2) BU Rollups----------------------------
def wavg(series, weights):
    s, w = series.astype(float), weights.fillna(0.0).astype(float)
    return (s*w).sum()/w.sum() if w.sum()>0 else np.nan

agg = (merged.groupby("business_unit")
       .agg(
           suppliers=("supplier_id","nunique"),
           countries=("iso3","nunique"),
           spend_total=("spend","sum"),
           avg_risk=("composite_risk_v2","mean"),
           avg_proba=("proba_ens","mean"),
           avg_anom=("anom_ens","mean"),
           severe_cnt=("risk_bucket", lambda s: (s=="Severe").sum()),
           high_cnt=("risk_bucket", lambda s: (s=="High").sum()),
           moderate_cnt=("risk_bucket", lambda s: (s=="Moderate").sum()),
           low_cnt=("risk_bucket", lambda s: (s=="Low").sum()),
           verylow_cnt=("risk_bucket", lambda s: (s=="Very Low").sum()),
           spend_at_risk=("spend_at_risk","sum")
       )
       .reset_index()
      )

waggs = merged.groupby("business_unit").apply(
    lambda g: pd.Series({
        "wavg_risk": wavg(g["composite_risk_v2"], g["spend"]),
        "wavg_proba": wavg(g["proba_ens"], g["spend"]),
        "wavg_anom": wavg(g["anom_ens"], g["spend"])
    })
).reset_index()

bu = agg.merge(waggs, on="business_unit", how="left")
bu["risk_flag"] = np.where(bu["wavg_risk"]>=0.3,"High attention","Stable")
bu.to_csv(BUART/"bu_table.csv",index=False)

# ----------------------------3) Visual BI Dashboards----------------------------
fig_bu = px.bar(bu, x="business_unit", y="wavg_risk", color="risk_flag",
                title="Spend-weighted Composite Risk by BU",
                hover_data=["suppliers","countries","spend_total","wavg_proba","wavg_anom"])
fig_bu.update_layout(height=400)
fig_bu.write_html(BUART/"bu_wavg_risk.html")

fig_bubble = px.scatter(bu, x="wavg_risk", y="wavg_anom", size="spend_total",
                        color="risk_flag", hover_name="business_unit",
                        title="BU Exposure (Spend vs Risk & Anomaly)")
fig_bubble.update_layout(height=500)
fig_bubble.write_html(BUART/"bu_exposure_bubble.html")

# ----------------------------4) BU Driver Decomposition----------------------------
def zscore(x): return (x-x.mean())/(x.std() if x.std()>0 else 1)
for c in ["news_7d_z_pm","cp_score","anom_ens","LPI","WGI_MEAN"]:
    if c not in merged.columns: merged[c] = np.nan
merged["_z_news"]=zscore(merged["news_7d_z_pm"])
merged["_z_cp"]=zscore(merged["cp_score"])
merged["_z_anom"]=zscore(merged["anom_ens"])
merged["_z_LPI"]=zscore(merged["LPI"])
merged["_z_WGI"]=zscore(merged["WGI_MEAN"])

def contrib(r):
    return {
        "Incident spike (z)": r["_z_news"],
        "Change-point (z)": 0.15*r["_z_cp"],
        "Anomaly (z)": 0.15*r["_z_anom"],
        "Governance (-)": -0.4*r["_z_WGI"],
        "Logistics (-)": -0.4*r["_z_LPI"],
        "Model proba": 0.25*r["proba_ens"]
    }

driver_out=[]
for bu_name,g in merged.groupby("business_unit"):
    w=g["spend"].fillna(0.0)
    if w.sum()==0: w=pd.Series(np.ones(len(g)),index=g.index)
    parts=[contrib(r) for _,r in g.iterrows()]
    d=pd.DataFrame(parts)
    weighted=(d.mul(w.values[:,None])).sum()/w.sum()
    weighted=weighted.sort_values(key=lambda s:s.abs(),ascending=False)
    driver_out.append({"business_unit":bu_name,**weighted.to_dict()})
drv=pd.DataFrame(driver_out).fillna(0)
drv.to_csv(BUART/"bu_driver_contributions.csv",index=False)

# ----------------------------5) Safe Merge + Scorecards with Mini BI----------------------------

cols_for_merge=[c for c in ["business_unit","suppliers","countries",
                            "severe_cnt","high_cnt","moderate_cnt","low_cnt","verylow_cnt"]
                if c in agg.columns]
safe_merge=bu.merge(agg[cols_for_merge],on="business_unit",how="left")

for _,r in safe_merge.iterrows():
    bu_name=r["business_unit"]
    top_table=(merged[merged["business_unit"]==bu_name]
               .sort_values("composite_risk_v2",ascending=False)
               [["supplier_id","supplier_name","iso3","criticality","spend",
                 "composite_risk_v2","proba_ens","anom_ens","risk_bucket"]]
               .head(10).copy())
    top_table["spend"]=top_table["spend"].map(lambda v:f"{v:,.0f}")
    tbl_html=top_table.to_html(index=False,border=1)

    # Small risk gauge (plotly)
    gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=r.get("wavg_risk",0.0),
        title={'text':"Avg Risk"},
        gauge={'axis':{'range':[0,1]},
               'bar':{'color':"crimson" if r.get("wavg_risk",0)<0.3 else "orange"},
               'steps':[{'range':[0,0.3],'color':"#cce5ff"},
                        {'range':[0.3,0.6],'color':"#ffe5b4"},
                        {'range':[0.6,1.0],'color':"#f5b7b1"}]}))
    gauge_path=BUART/f"gauge_{bu_name}.html"
    gauge.write_html(gauge_path, include_plotlyjs='cdn', full_html=False)

    # Top 5 driver bar
    if bu_name in drv["business_unit"].values:
        d=drv[drv["business_unit"]==bu_name].drop(columns="business_unit").T
        d.columns=["impact"]
        d=d.sort_values("impact",key=lambda s:s.abs(),ascending=False).head(5)
        driver_fig=px.bar(d,x=d.index,y="impact",color="impact",title="Top 5 Drivers")
        driver_path=BUART/f"drivers_{bu_name}.html"
        driver_fig.write_html(driver_path, include_plotlyjs='cdn', full_html=False)
    else:
        driver_path=""

    html=f"""
    <html><head><meta charset='utf-8'><title>{bu_name} Scorecard</title></head>
    <body style='font-family:Arial;margin:20px;'>
    <h2>Business Unit Scorecard — {bu_name}</h2>
    <p><b>Generated:</b> {datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}</p>
    <h3>KPIs</h3>
    <ul>
      <li>Suppliers: {int(r.get('suppliers',len(top_table)))}</li>
      <li>Countries: {int(r.get('countries',top_table['iso3'].nunique()))}</li>
      <li>Spend total: {r.get('spend_total',0):,.0f}</li>
      <li>Spend-weighted risk: {r.get('wavg_risk',0):.2f}</li>
      <li>Spend-weighted anomaly: {r.get('wavg_anom',0):.2f}</li>
    </ul>
    <iframe src='../gauge_{bu_name}.html' width='400' height='250' frameborder='0'></iframe>
    <iframe src='../drivers_{bu_name}.html' width='500' height='300' frameborder='0'></iframe>
    <h3>Top Suppliers</h3>
    {tbl_html}
    <p><i>Charts and detailed visuals saved alongside this file.</i></p>
    </body></html>
    """
    (SCARDS/f"scorecard_{bu_name}.html").write_text(html,encoding="utf-8")

# ----------------------------6) Exports Summary----------------------------

print("==== BU BI Pack — Outputs ====")
print("• BU KPIs:", (BUART/'bu_table.csv'))
print("• Supplier join:", (BUART/'suppliers_with_risk.csv'))
print("• Driver contributions:", (BUART/'bu_driver_contributions.csv'))
print("• Scorecards:", SCARDS)
print("• Visuals: bar, bubble, heatmap, and mini dashboards inside /artifacts/bu/")

import shutil
import os
from google.colab import files

zip_path = '/content/live_project/artifacts/bu.zip'
directory_to_zip = '/content/live_project/artifacts/bu'

# Create the zip file
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', directory_to_zip)

# Offer the file for download
files.download(zip_path)


# Economic assumptions (you can tune)
ECON = {
    "Cost_false_alert":   10_000.0,   # analyst time + noise cost per FP
    "Cost_missed_event": 200_000.0,   # line-down/stockout $ impact per FN (conservative)
    "Benefit_caught":     80_000.0,   # avoided expediting/reroute etc per TP
    "Lead_time_days_gain": 5          # typical lead-time advantage if flagged early
}

TOPK = 5                 # top-K for backtest precision@K / recall@K
WEAK_POS_QUANT = 0.85    # weak label quantile if needed
THRESH_GRID = np.round(np.linspace(0.05, 0.95, 19), 2)

risk_p   = GOLD/"country_risk_composite_live.csv"
alerts_p = GOLD/"alerts.json"

if not risk_p.exists():
    raise FileNotFoundError(f"Missing {risk_p}. Run main pipeline first.")

risk = pd.read_csv(risk_p)
risk["iso3"] = risk["iso3"].astype(str)
# graceful defaults
for c, default in [
    ("proba_ens", np.nan), ("composite_risk_v2", np.nan), ("news_7d_z_pm", 0.0),
    ("cp_score", 0.0), ("anom_ens", 0.0), ("LPI", np.nan), ("WGI_MEAN", np.nan)
]:
    if c not in risk.columns: risk[c] = default

alerts = {}
if alerts_p.exists():
    try:
        alerts = json.loads(alerts_p.read_text())
    except Exception:
        alerts = {}

# Weak labels (if not provided): top (1-quantile) by composite_v2
if "weak_label" not in risk.columns:
    qthr = risk["composite_risk_v2"].quantile(WEAK_POS_QUANT)
    risk["weak_label"] = (risk["composite_risk_v2"] >= qthr).astype(int)


def confusion_from_threshold(p, y, thr):
    yhat = (p >= thr).astype(int)
    tp = int(((y==1)&(yhat==1)).sum())
    fp = int(((y==0)&(yhat==1)).sum())
    fn = int(((y==1)&(yhat==0)).sum())
    tn = int(((y==0)&(yhat==0)).sum())
    return tp, fp, fn, tn

def profit(tp, fp, fn, econ):
    return tp*econ["Benefit_caught"] - fp*econ["Cost_false_alert"] - fn*econ["Cost_missed_event"]

y = risk["weak_label"].values
p = risk["proba_ens"].fillna(0.0).values

roi = []
for thr in THRESH_GRID:
    tp, fp, fn, tn = confusion_from_threshold(p, y, thr)
    pr = profit(tp, fp, fn, ECON)
    roi.append({"threshold": float(thr), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                "profit": float(pr)})

roi_df = pd.DataFrame(roi).sort_values("threshold")
roi_df.to_csv(UPG/"roi_curve.csv", index=False)

fig_roi = px.line(roi_df, x="threshold", y="profit",
                  title="ROI vs Threshold (current snapshot)",
                  markers=True)
fig_roi.update_layout(height=400)
fig_roi.write_html(UPG/"roi_curve.html")

# Sensitivity tornado: vary each econ knob ±30%
def roi_with_econ(econ_overrides):
    econ = {**ECON, **econ_overrides}
    # pick current best thr from base case
    base_best_thr = roi_df.loc[roi_df["profit"].idxmax(), "threshold"]
    tp, fp, fn, tn = confusion_from_threshold(p, y, base_best_thr)
    return profit(tp, fp, fn, econ)

sens = []
for k in ["Benefit_caught","Cost_false_alert","Cost_missed_event"]:
    base = roi_with_econ({})
    up   = roi_with_econ({k: ECON[k]*1.3})
    dn   = roi_with_econ({k: ECON[k]*0.7})
    sens.append({"factor": k, "delta_up": float(up-base), "delta_down": float(dn-base), "base_profit": float(base)})
sens_df = pd.DataFrame(sens)
sens_df.to_csv(UPG/"roi_sensitivity_tornado.csv", index=False)

fig_tornado = go.Figure()
for _, r in sens_df.iterrows():
    fig_tornado.add_trace(go.Bar(
        x=[r["delta_up"]],
        y=[r["factor"]],
        orientation='h',
        name="+30%",
        marker_color="seagreen"
    ))
    fig_tornado.add_trace(go.Bar(
        x=[r["delta_down"]],
        y=[r["factor"]],
        orientation='h',
        name="-30%",
        marker_color="indianred"
    ))
fig_tornado.update_layout(barmode='relative', title="ROI Sensitivity (Tornado, base thr)", height=420)
fig_tornado.write_html(UPG/"roi_sensitivity_tornado.html")


def list_history():
    if not HISTORY.exists(): return []
    out=[]
    for d in sorted([p for p in HISTORY.iterdir() if p.is_dir()]):
        f = d/"country_risk_composite_live.csv"
        if f.exists(): out.append(f)
    return out

hist_files = list_history()
backtest = []
if len(hist_files) >= 3:
    # Use monthly “next-period” event = top X% risk in next snapshot
    # Predict at time t using proba_ens_t; evaluate against events at t+1
    for i in range(len(hist_files)-1):
        f_t   = hist_files[i]
        f_t1  = hist_files[i+1]
        date_t  = f_t.parent.name
        date_t1 = f_t1.parent.name
        df_t  = pd.read_csv(f_t)
        df_t1 = pd.read_csv(f_t1)
        # ensure cols
        for c in ["iso3","proba_ens","composite_risk_v2"]:
            if c not in df_t.columns: df_t[c] = np.nan
            if c not in df_t1.columns: df_t1[c] = np.nan
        df_t["iso3"] = df_t["iso3"].astype(str)
        df_t1["iso3"] = df_t1["iso3"].astype(str)

        # Define "event@t1" as top quantile of composite at t+1
        qthr = df_t1["composite_risk_v2"].quantile(WEAK_POS_QUANT)
        events = (df_t1.set_index("iso3")["composite_risk_v2"] >= qthr).astype(int)

        # predictions at t: top-K by proba_ens
        preds = df_t[["iso3","proba_ens"]].dropna().sort_values("proba_ens", ascending=False)
        topk = preds.head(TOPK)["iso3"].tolist()

        # Compute precision@K, recall@K
        y_true = events.reindex(preds["iso3"]).fillna(0).astype(int)
        k = min(TOPK, len(preds))
        precK = float(y_true.loc[topk].mean()) if k>0 else np.nan
        # recallK = (#events caught in topK) / (#events total)
        total_events = int(events.sum())
        recallK = float(y_true.loc[topk].sum()/total_events) if total_events>0 and k>0 else np.nan

        backtest.append({"t": date_t, "t_plus": date_t1, "precision@K": precK, "recall@K": recallK,
                         "K": int(TOPK), "events_total": int(total_events)})

bt_df = pd.DataFrame(backtest)
if not bt_df.empty:
    bt_df.to_csv(UPG/"rolling_backtest_prek_reck.csv", index=False)
    fig_bt = px.line(bt_df, x="t", y=["precision@K","recall@K"], markers=True,
                     title=f"Rolling Back-test (K={TOPK})")
    fig_bt.update_layout(height=420)
    fig_bt.write_html(UPG/"rolling_backtest_prek_reck.html")


# ----------------------------4) Ablation (drop feature blocks from current composite)----------------------------

def z_s(col):
    s = col.astype(float)
    sd = s.std()
    return (s - s.mean()) / (sd if sd and sd>0 else 1.0)

risk["_z_news"] = z_s(risk["news_7d_z_pm"].fillna(0))
risk["_z_cp"]   = z_s(risk["cp_score"].fillna(0))
risk["_z_anom"] = z_s(risk["anom_ens"].fillna(0))
risk["_z_LPI"]  = z_s(risk["LPI"].fillna(risk["LPI"].median() if pd.notna(risk["LPI"].median()) else 0))
risk["_z_WGI"]  = z_s(risk["WGI_MEAN"].fillna(risk["WGI_MEAN"].median() if pd.notna(risk["WGI_MEAN"].median()) else 0))

def composite_full(df):
    return ( 1.0*df["_z_news"] + 0.25*df["proba_ens"].fillna(0)
           + 0.15*df["_z_cp"]  - 0.4*df["_z_LPI"] - 0.4*df["_z_WGI"]
           + 0.15*df["_z_anom"] )

ablations = {}
base = composite_full(risk).values
ablations["base"] = base

ablations["drop_news"]  = (0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values
ablations["drop_cp"]    = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values
ablations["drop_anom"]  = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"]).values
ablations["drop_LPI"]   = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values
ablations["drop_WGI"]   = (1.0*risk["_z_news"] + 0.25*risk["proba_ens"].fillna(0) + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] + 0.15*risk["_z_anom"]).values
ablations["drop_proba"] = (1.0*risk["_z_news"] + 0.15*risk["_z_cp"] - 0.4*risk["_z_LPI"] - 0.4*risk["_z_WGI"] + 0.15*risk["_z_anom"]).values

# Evaluate delta in rank correlation vs base (Spearman)
from scipy.stats import spearmanr
rows=[]
for name, vals in ablations.items():
    rho = float(spearmanr(base, vals, nan_policy="omit").correlation)
    rows.append({"ablation": name, "spearman_to_base": rho})
abl_df = pd.DataFrame(rows).sort_values("spearman_to_base", ascending=False)
abl_df.to_csv(UPG/"ablation_rank_stability.csv", index=False)
fig_abl = px.bar(abl_df, x="ablation", y="spearman_to_base",
                 title="Ablation: Rank Stability vs Base Composite")
fig_abl.update_layout(height=420)
fig_abl.write_html(UPG/"ablation_rank_stability.html")

# Optional: ROI delta if we re-threshold on ablated score (using weak labels)
def roi_from_score(score):
    local = []
    for thr in THRESH_GRID:
        yhat = (score >= thr).astype(int)
        tp = int(((y==1)&(yhat==1)).sum())
        fp = int(((y==0)&(yhat==1)).sum())
        fn = int(((y==1)&(yhat==0)).sum())
        local.append({"thr": float(thr), "profit": profit(tp, fp, fn, ECON)})
    ldf = pd.DataFrame(local)
    return float(ldf["profit"].max())

roi_rows=[]
for name, vals in ablations.items():
    roi_rows.append({"score": name, "best_profit": roi_from_score(vals)})
pd.DataFrame(roi_rows).to_csv(UPG/"ablation_roi.csv", index=False)

# ----------------------------5) Stress Tests (±10/20/30% perturbations)----------------------------
def stress_score(df, pct):
    out = df.copy()
    out["_z_news_s"] = z_s(df["news_7d_z_pm"].fillna(0)*(1+pct))
    out["_z_cp_s"]   = z_s(df["cp_score"].fillna(0)*(1+pct))
    out["_z_anom_s"] = z_s(df["anom_ens"].fillna(0)*(1+pct))
    out["_z_LPI_s"]  = z_s(df["LPI"].fillna(0)*(1-pct))  # inverse for “-” terms = conservative
    out["_z_WGI_s"]  = z_s(df["WGI_MEAN"].fillna(0)*(1-pct))
    return ( 1.0*out["_z_news_s"] + 0.25*out["proba_ens"].fillna(0)
           + 0.15*out["_z_cp_s"]  - 0.4*out["_z_LPI_s"] - 0.4*out["_z_WGI_s"]
           + 0.15*out["_z_anom_s"] ).values

base_score = composite_full(risk).values
stress = []
for pct in [0.1, 0.2, 0.3]:
    s = stress_score(risk, pct)
    rho = float(spearmanr(base_score, s, nan_policy="omit").correlation)
    # stability of alert set at current best threshold
    best_thr = roi_df.loc[roi_df["profit"].idxmax(), "threshold"]
    base_set = set(risk.loc[base_score>=best_thr,"iso3"])
    stress_set = set(risk.loc[s>=best_thr,"iso3"])
    jacc = (len(base_set & stress_set) / len(base_set | stress_set)) if (base_set or stress_set) else 1.0
    stress.append({"pct": pct, "spearman_to_base": rho, "alert_jaccard": float(jacc)})
stress_df = pd.DataFrame(stress)
stress_df.to_csv(UPG/"stress_stability.csv", index=False)
fig_str = px.bar(stress_df.melt(id_vars="pct"), x="pct", y="value", color="variable",
                 title="Stress Test: Rank/Alert Set Stability", barmode="group")
fig_str.update_layout(height=420)
fig_str.write_html(UPG/"stress_stability.html")

# ----------------------------6) Causality-ish & Bias Checks----------------------------
# Placebo: shuffle iso3 mapping, recompute corr
shuf = risk.copy()
shuf = shuf.sample(frac=1.0, random_state=42).reset_index(drop=True)
rho_placebo = float(spearmanr(base_score, composite_full(shuf).values, nan_policy="omit").correlation)
json.dump({"placebo_spearman": rho_placebo}, open(UPG/"placebo_check.json","w"), indent=2)

# Bias stratification by structure tiers
def tier_by(col, q=[0.33, 0.66]):
    s = risk[col]
    try:
        b = s.quantile(q).values
    except Exception:
        b = [s.median(), s.median()]
    def lab(v):
        if pd.isna(v): return "Unknown"
        if v <= b[0]: return "Low"
        if v <= b[1]: return "Mid"
        return "High"
    return s.apply(lab)

risk["tier_LPI"] = tier_by("LPI")
risk["tier_WGI"] = tier_by("WGI_MEAN")

bias_rows=[]
for dim in ["tier_LPI","tier_WGI"]:
    for lvl, g in risk.groupby(dim):
        if len(g)==0: continue
        # at best_thr from base ROI
        best_thr = roi_df.loc[roi_df["profit"].idxmax(), "threshold"]
        tp, fp, fn, tn = confusion_from_threshold(g["proba_ens"].fillna(0).values,
                                                  g["weak_label"].values, best_thr)
        bias_rows.append({
            "dimension": dim, "level": str(lvl),
            "n": int(len(g)),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn
        })
bias_df = pd.DataFrame(bias_rows)
bias_df.to_csv(UPG/"bias_stratified_confusion.csv", index=False)

# ----------------------------7) Model Card & Governance----------------------------
model_card = {
    "generated_at": datetime.utcnow().isoformat()+"Z",
    "objective": "Early detection of country-level supply chain risk using news-based signals + structure.",
    "sources": ["GDELT timelines (30d live)", "World Bank LPI/WGI", "Population (per-million)"],
    "models": ["RF, LightGBM, XGBoost, CatBoost (stacked)", "Anomaly ensemble (IF, OCSVM, LOF, HDBSCAN)"],
    "composite": "1.0*z(news spike) + 0.25*proba_ens + 0.15*z(cp) - 0.4*z(LPI) - 0.4*z(WGI) + 0.15*z(anom)",
    "refresh_cadence": "Daily for GDELT; annual for LPI/WGI (latest available).",
    "monitoring": {
        "drift": ["news intensity drift", "class balance", "threshold stability"],
        "alerts": ["Severe/High/Moderate buckets with playbook"]
    },
    "known_limits": [
        "Media bias/coverage bias by country",
        "Weak labels proxy true disruptions",
        "Short horizon on live stream (extend with history for strategy)"
    ],
    "econ_assumptions": ECON,
    "artifacts": {
        "roi_curve_csv": (UPG/"roi_curve.csv").as_posix(),
        "roi_curve_html": (UPG/"roi_curve.html").as_posix(),
        "roi_sensitivity_html": (UPG/"roi_sensitivity_tornado.html").as_posix(),
        "rolling_backtest_html": (UPG/"rolling_backtest_prek_reck.html").as_posix() if (UPG/"rolling_backtest_prek_reck.html").exists() else None,
        "ablation_html": (UPG/"ablation_rank_stability.html").as_posix(),
        "stress_html": (UPG/"stress_stability.html").as_posix(),
        "bias_csv": (UPG/"bias_stratified_confusion.csv").as_posix()
    }
}
(UPG/"model_card.json").write_text(json.dumps(model_card, indent=2))

# Pretty Markdown for faculty / stakeholders
mc_md = f"""# Model Card — Live Supply-Chain Risk

**Generated:** {model_card['generated_at']}

## Objective
Early detection of country-level supply chain risk to prioritize due diligence and mitigation.

## Data & Signals
- **GDELT** news timelines (live 30d; extendable)
- **World Bank** LPI & WGI (structure)
- Population normalization (per-million)

## Model Stack
- RF, LightGBM, XGBoost, CatBoost (stacked)
- Anomaly ensemble: IsolationForest, OneClassSVM, LOF, HDBSCAN
- Composite: `1.0*z(news) + 0.25*proba + 0.15*z(cp) - 0.4*z(LPI) - 0.4*z(WGI) + 0.15*z(anom)`

## Validation & Ops
- ROI curves & sensitivity tornado
- Rolling back-test (if history exists)
- Feature ablation & stress tests
- Bias stratification by LPI/WGI tiers
- Drift monitors: news intensity, class balance, threshold stability

## Known Limits
Media bias, weak labels proxy, structural data refresh lag.

## Economics (assumptions)
{json.dumps(ECON, indent=2)}

## Key Artifacts
- ROI curve: { (UPG/'roi_curve.html').as_posix() }
- Sensitivity tornado: { (UPG/'roi_sensitivity_tornado.html').as_posix() }
- Ablation: { (UPG/'ablation_rank_stability.html').as_posix() }
- Stress stability: { (UPG/'stress_stability.html').as_posix() }
- Back-test: { (UPG/'rolling_backtest_prek_reck.html').as_posix() if (UPG/'rolling_backtest_prek_reck.html').exists() else '—' }
- Bias CSV: { (UPG/'bias_stratified_confusion.csv').as_posix() }
"""
(UPG/"model_card.md").write_text(mc_md, encoding="utf-8")

# ----------------------------8) Productization Stubs (scheduler + API)----------------------------
# a) Cron/GitHub Actions style shell template
cron_sh = """#!/usr/bin/env bash
set -e
# Example: schedule this script daily with cron or GitHub Actions runner
# 1) Run main pipeline (python or colab nbconvert)
# 2) Run BI packs
# 3) Run upgrade pack
# 4) Sync artifacts to object storage (S3/GCS) or BI server

python main_pipeline.py
python bi_pack.py
python bu_bi_pack.py
python upgrade_pack.py

# rsync or aws s3 cp artifacts
# aws s3 sync /content/live_project/artifacts s3://your-bucket/live_project/artifacts --delete
"""
(UPG/"cron_template.sh").write_text(cron_sh)

# b) Minimal FastAPI stub (serves alerts & KPIs)
fastapi_stub = """from fastapi import FastAPI
from fastapi.responses import JSONResponse
import json, pathlib

BASE = pathlib.Path('/content/live_project')
GOLD = BASE/'data'/'gold'
ART  = BASE/'artifacts'
UPG  = ART/'upgrade'

app = FastAPI(title='Supply Chain Risk API', version='0.1')

def load_json(p):
    if p.exists():
        try: return json.loads(p.read_text())
        except: return {}
    return {}

@app.get('/health')
def health(): return {'ok': True}

@app.get('/alerts')
def alerts():
    return JSONResponse(load_json(GOLD/'alerts.json'))

@app.get('/kpis')
def kpis():
    return JSONResponse(load_json(ART/'bi_kpis.json'))

@app.get('/model_card')
def model_card():
    return JSONResponse(load_json(UPG/'model_card.json'))
"""
(UPG/"fastapi_stub.py").write_text(fastapi_stub)

# ----------------------------9) Console Summary----------------------------
print("==== Upgrade Pack — Artifacts ====")
print("• ROI curve CSV/HTML:", (UPG/"roi_curve.csv").as_posix(), "|", (UPG/"roi_curve.html").as_posix())
print("• ROI sensitivity (tornado):", (UPG/"roi_sensitivity_tornado.html").as_posix())
if (UPG/"rolling_backtest_prek_reck.html").exists():
    print("• Rolling back-test:", (UPG/"rolling_backtest_prek_reck.html").as_posix())
else:
    print("• Rolling back-test: (skipped — add history snapshots under /content/live_project/history/YYYY-MM-DD/)")
print("• Ablation rank stability:", (UPG/"ablation_rank_stability.html").as_posix())
print("• Stress stability:", (UPG/"stress_stability.html").as_posix())
print("• Bias stratification CSV:", (UPG/"bias_stratified_confusion.csv").as_posix())
print("• Model card JSON/MD:", (UPG/"model_card.json").as_posix(), "|", (UPG/"model_card.md").as_posix())
print("• Productization: cron_template.sh, fastapi_stub.py in", UPG.as_posix())

# Friendly hints
print("\nTips:")
print(" - To enable rolling back-test, create monthly folders like /history/2025-01-01/country_risk_composite_live.csv (and so on).")
print(" - Tune ECON in this cell to your company’s costs; the tornado will update.")
print(" - Serve alerts & KPIs: `uvicorn artifacts/upgrade/fastapi_stub:app --reload --host 0.0.0.0 --port 8000`")

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit


Weak-label metrics (in-sample; higher PR/ROC, lower Brier is better):
{'lgbm_proba': {'brier': 2.1929204332090378e-07, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'proba_ens': {'brier': 0.00497281166882194, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'rf_proba': {'brier': 0.0032906666666666674, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'xgb_proba': {'brier': 0.02748181521513848, 'pr_auc': 1.0, 'roc_auc': 1.0}}

In-sample profit-optimal probability threshold: 0.15 (proxy profit=3200.0)

Out-of-fold metrics (more realistic estimate):
{'ens': {'brier': 0.04579623893994122,
         'pr_auc': 0.9166666666666666,
         'roc_auc': 0.9761904761904762},
 'lgbm': {'brier': 0.07896300033429134,
          'pr_auc': 0.6625,
          'roc_auc': 0.9226190476190477},
 'rf': {'brier': 0.04280398712105383, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'xgb': {'brier': 0.07523785500909742,
         'pr_auc': 0.6083333333333333,
         'roc_auc': 0.9404761904761905}}

OOF profit-optimal probability threshold: 0.30 (proxy profit=3180.0)
====

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

==== Upgrade Pack — Artifacts ====
• ROI curve CSV/HTML: /content/live_project/artifacts/upgrade/roi_curve.csv | /content/live_project/artifacts/upgrade/roi_curve.html
• ROI sensitivity (tornado): /content/live_project/artifacts/upgrade/roi_sensitivity_tornado.html
• Rolling back-test: (skipped — add history snapshots under /content/live_project/history/YYYY-MM-DD/)
• Ablation rank stability: /content/live_project/artifacts/upgrade/ablation_rank_stability.html
• Stress stability: /content/live_project/artifacts/upgrade/stress_stability.html
• Bias stratification CSV: /content/live_project/artifacts/upgrade/bias_stratified_confusion.csv
• Model card JSON/MD: /content/live_project/artifacts/upgrade/model_card.json | /content/live_project/artifacts/upgrade/model_card.md
• Productization: cron_template.sh, fastapi_stub.py in /content/live_project/artifacts/upgrade

Tips:
 - To enable rolling back-test, create monthly folders like /history/2025-01-01/country_risk_composite_live.csv (and so 

...

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit


Weak-label metrics (in-sample; higher PR/ROC, lower Brier is better):
{'lgbm_proba': {'brier': 2.1929204332090378e-07, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'proba_ens': {'brier': 0.00497281166882194, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'rf_proba': {'brier': 0.0032906666666666674, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'xgb_proba': {'brier': 0.02748181521513848, 'pr_auc': 1.0, 'roc_auc': 1.0}}

In-sample profit-optimal probability threshold: 0.15 (proxy profit=3200.0)

Out-of-fold metrics (more realistic estimate):
{'ens': {'brier': 0.04579623893994122,
         'pr_auc': 0.9166666666666666,
         'roc_auc': 0.9761904761904762},
 'lgbm': {'brier': 0.07896300033429134,
          'pr_auc': 0.6625,
          'roc_auc': 0.9226190476190477},
 'rf': {'brier': 0.04280398712105383, 'pr_auc': 1.0, 'roc_auc': 1.0},
 'xgb': {'brier': 0.07523785500909742,
         'pr_auc': 0.6083333333333333,
         'roc_auc': 0.9404761904761905}}

OOF profit-optimal probability threshold: 0.30 (proxy profit=3180.0)
====

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

==== Upgrade Pack — Artifacts ====
• ROI curve CSV/HTML: /content/live_project/artifacts/upgrade/roi_curve.csv | /content/live_project/artifacts/upgrade/roi_curve.html
• ROI sensitivity (tornado): /content/live_project/artifacts/upgrade/roi_sensitivity_tornado.html
• Rolling back-test: (skipped — add history snapshots under /content/live_project/history/YYYY-MM-DD/)
• Ablation rank stability: /content/live_project/artifacts/upgrade/ablation_rank_stability.html
• Stress stability: /content/live_project/artifacts/upgrade/stress_stability.html
• Bias stratification CSV: /content/live_project/artifacts/upgrade/bias_stratified_confusion.csv
• Model card JSON/MD: /content/live_project/artifacts/upgrade/model_card.json | /content/live_project/artifacts/upgrade/model_card.md
• Productization: cron_template.sh, fastapi_stub.py in /content/live_project/artifacts/upgrade

Tips:
 - To enable rolling back-test, create monthly folders like /history/2025-01-01/country_risk_composite_live.csv (and so 

In [ ]:


import json
import pathlib
import pandas as pd

BASE   = pathlib.Path("/content/live_project")
GOLD   = BASE / "data" / "gold"
ART    = BASE / "artifacts"
UPG    = ART / "upgrade"


def _load(path, kind="csv"):
    if not path.exists():
        return None
    if kind == "csv":
        return pd.read_csv(path)
    if kind == "json":
        return json.loads(path.read_text())
    return None


def describe_use_cases():
    """
    Print 5 concrete use cases, using the actual artifacts
    produced by the main pipeline as evidence.
    """
    risk = _load(GOLD / "country_risk_composite_live.csv", "csv")
    alerts = _load(GOLD / "alerts.json", "json")
    bu = _load(ART / "bu" / "bu_table.csv", "csv")
    kpis = _load(ART / "bi_kpis.json", "json")
    roi = _load(UPG / "roi_curve.csv", "csv")

    print("==== 5 PROOF‑OF‑VALUE USE CASES ====\n")

    # 1) Daily country watchlist for compliance / Exec briefings
    print("1) Daily Country Watchlist (Compliance / Exec Briefings)")
    if risk is not None:
        top = risk.sort_values("composite_risk_v2", ascending=False).head(5)
        print("   The model ranks countries by live composite risk. Example top‑5 today:")
        for _, r in top.iterrows():
            print(f"   - {r['iso3']}: risk={r['composite_risk_v2']:.2f}, "
                  f"proba={r.get('proba_ens', float('nan')):.2f}, "
                  f"anom={r.get('anom_ens', float('nan')):.2f}")
    else:
        print("   (risk table missing – run main pipeline first.)")
    print("   ➜ This can be dropped straight into weekly risk reports or dashboards.\n")

    # 2) Alert triage with economics (where to spend analyst time)
    print("2) Alert Triage with Economics (Analyst Time Prioritisation)")
    if alerts is not None:
        thr = alerts.get("threshold_prob_oof", alerts.get("threshold_prob_in_sample"))
        profit = alerts.get("profit_proxy_oof", alerts.get("profit_proxy_in_sample"))
        hits = alerts.get("hits", [])[:5]
        print(f"   Optimised alert threshold (OOF): {thr:.2f} with proxy profit ≈ {profit:.1f}.")
        print("   Top alert hits at this threshold:")
        for h in hits:
            print(f"   - {h['iso3']}: composite={h['composite_risk_v2']:.2f}, "
                  f"proba={h.get('proba_ens', float('nan')):.2f}")
    else:
        print("   (alerts.json missing – run main pipeline first.)")
    print("   ➜ Shows we can turn scores into cost‑aware, actionable alert queues.\n")

    # 3) BU‑level exposure and scorecards for budget / portfolio reviews
    print("3) BU‑Level Exposure & Scorecards (Budget / Portfolio Reviews)")
    if bu is not None and not bu.empty:
        top_bu = bu.sort_values("wavg_risk", ascending=False).head(3)
        print("   Spend‑weighted risk by business unit (top‑3 by wavg_risk):")
        for _, r in top_bu.iterrows():
            print(f"   - {r['business_unit']}: wavg_risk={r['wavg_risk']:.2f}, "
                  f"spend_total={r['spend_total']:,.0f}, suppliers={int(r['suppliers'])}")
    else:
        print("   (BU table missing – run BU pack in main cell.)")
    print("   ➜ Lets BU owners see ‘where to look first’ and supports quarterly reviews.\n")

    # 4) BI / analytics integration for dashboards
    print("4) BI & Analytics Integration (Power BI / Tableau / Looker)")
    if kpis is not None:
        port = kpis.get("portfolio", {})
        print("   Portfolio KPIs exported to JSON for direct BI consumption:")
        print(f"   - n_countries = {port.get('n_countries')}")
        print(f"   - avg_composite_risk = {port.get('avg_composite_risk'):.3f}")
        print(f"   - avg_model_proba    = {port.get('avg_proba_ens'):.3f}")
        print(f"   - avg_anomaly        = {port.get('avg_anomaly'):.3f}")
    else:
        print("   (bi_kpis.json missing – run BI add‑on in main cell.)")
    print("   ➜ Proves the pipeline is BI‑ready, not just a notebook experiment.\n")

    # 5) ROI & governance narrative for management / audit
    print("5) ROI & Governance Evidence (Management / Audit / Model Risk)")
    if roi is not None and not roi.empty:
        best_row = roi.loc[roi["profit"].idxmax()]
        print(f"   ROI curve shows best threshold ≈ {best_row['threshold']:.2f} "
              f"with proxy profit ≈ {best_row['profit']:.1f}.")
        print("   Upgrade pack also writes:")
        print(f"   - Model card: { (UPG/'model_card.json').as_posix() }")
        print(f"   - Bias diagnostics: { (UPG/'bias_stratified_confusion.csv').as_posix() }")
        print(f"   - Stress tests: { (UPG/'stress_stability.html').as_posix() }")
    else:
        print("   (roi_curve.csv missing – run upgrade pack in main cell.)")
    print("   ➜ Demonstrates not just accuracy, but economics, robustness and governance.\n")

    print("==== END USE‑CASE SUMMARY ====")


describe_use_cases()



==== 5 PROOF‑OF‑VALUE USE CASES ====

1) Daily Country Watchlist (Compliance / Exec Briefings)
   The model ranks countries by live composite risk. Example top‑5 today:
   - MEX: risk=2.39, proba=0.88, anom=0.31
   - BRA: risk=1.69, proba=0.88, anom=0.21
   - TUR: risk=1.38, proba=0.83, anom=0.19
   - VNM: risk=1.36, proba=0.86, anom=0.17
   - IND: risk=0.88, proba=0.12, anom=0.15
   ➜ This can be dropped straight into weekly risk reports or dashboards.

2) Alert Triage with Economics (Analyst Time Prioritisation)
   Optimised alert threshold (OOF): 0.30 with proxy profit ≈ 3180.0.
   Top alert hits at this threshold:
   - MEX: composite=2.39, proba=0.88
   - BRA: composite=1.69, proba=0.88
   - TUR: composite=1.38, proba=0.83
   - VNM: composite=1.36, proba=0.86
   ➜ Shows we can turn scores into cost‑aware, actionable alert queues.

3) BU‑Level Exposure & Scorecards (Budget / Portfolio Reviews)
   Spend‑weighted risk by business unit (top‑3 by wavg_risk):
   - Logistics: wavg_risk=1.

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))

get_ipython().events.register('pre_run_cell', set_css)